# Dataset profiling

In [ ]:
# ============================================================
# DATASET PROFILING (ONE CELL) — Recod.ai/LUC Scientific Image Forgery
# REVISI FULL (Fail-safe + Duplicate-safe + HWK/KHW mask-safe + better root detection)
#
# DINO_DIR fixed:
#   /kaggle/input/dinov2/pytorch/base/1
#
# Output:
# - /kaggle/working/recodai_luc_prof/image_profile.parquet
# - /kaggle/working/recodai_luc_prof/mask_index.parquet
# - /kaggle/working/recodai_luc_prof/mask_profile.parquet
# - /kaggle/working/recodai_luc_prof/paths.json
# ============================================================

import os, re, json, math, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore", category=FutureWarning)

# ----------------------------
# 0) Fixed DINO path
# ----------------------------
DINO_DIR = Path("/kaggle/input/dinov2/pytorch/base/1")
need_files = ["config.json", "pytorch_model.bin", "preprocessor_config.json"]
dino_ok = DINO_DIR.exists() and all((DINO_DIR / f).exists() for f in need_files)
print("DINO_DIR:", str(DINO_DIR), "| OK:", bool(dino_ok))
if not dino_ok:
    missing = [f for f in need_files if not (DINO_DIR / f).exists()]
    print("  Missing:", missing)

# ----------------------------
# 1) Auto-detect COMP_ROOT (robust scoring)
# ----------------------------
def score_comp_root(root: Path):
    s = 0
    if (root / "sample_submission.csv").exists(): s += 5
    if (root / "train_images").exists(): s += 3
    if (root / "test_images").exists(): s += 3
    if (root / "train_masks").exists(): s += 2
    # prefer expected subfolders
    if (root / "train_images" / "authentic").exists(): s += 2
    if (root / "train_images" / "forged").exists(): s += 2
    # supplemental presence
    if (root / "supplemental_images").exists(): s += 1
    if (root / "supplemental_masks").exists(): s += 1
    # small tie-break: shorter path better
    s2 = -len(str(root))
    return (s, s2)

def find_comp_root():
    base = Path("/kaggle/input")
    if not base.exists():
        raise FileNotFoundError("Not on Kaggle: /kaggle/input not found.")
    cands = []

    # direct children
    for d in base.iterdir():
        if not d.is_dir():
            continue
        if (d / "sample_submission.csv").exists() and (d / "train_images").exists() and (d / "test_images").exists():
            cands.append(d)

    # fallback: search for sample_submission then validate siblings
    if not cands:
        for d in base.iterdir():
            if not d.is_dir():
                continue
            for ss in d.rglob("sample_submission.csv"):
                root = ss.parent
                if (root / "train_images").exists() and (root / "test_images").exists():
                    cands.append(root)
                    break

    if not cands:
        raise FileNotFoundError("Cannot find competition root under /kaggle/input")

    cands = sorted(set(map(Path, cands)), key=lambda x: score_comp_root(x), reverse=True)
    best = cands[0]
    return best

COMP_ROOT = find_comp_root()

TRAIN_IMG_DIR = COMP_ROOT / "train_images"
TRAIN_IMG_AUTH = TRAIN_IMG_DIR / "authentic"
TRAIN_IMG_FORG = TRAIN_IMG_DIR / "forged"
TRAIN_MASK_DIR = COMP_ROOT / "train_masks"

SUP_IMG_DIR = COMP_ROOT / "supplemental_images"
SUP_MASK_DIR = COMP_ROOT / "supplemental_masks"

TEST_IMG_DIR = COMP_ROOT / "test_images"
SAMPLE_SUB = COMP_ROOT / "sample_submission.csv"

OUT_DIR = Path("/kaggle/working/recodai_luc_prof")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "COMP_ROOT": str(COMP_ROOT),
    "SAMPLE_SUB": str(SAMPLE_SUB),
    "TRAIN_IMG_AUTH": str(TRAIN_IMG_AUTH),
    "TRAIN_IMG_FORG": str(TRAIN_IMG_FORG),
    "TRAIN_MASK_DIR": str(TRAIN_MASK_DIR),
    "SUP_IMG_DIR": str(SUP_IMG_DIR),
    "SUP_MASK_DIR": str(SUP_MASK_DIR),
    "TEST_IMG_DIR": str(TEST_IMG_DIR),
    "DINO_DIR": str(DINO_DIR),
    "OUT_DIR": str(OUT_DIR),
}
(OUT_DIR / "paths.json").write_text(json.dumps(PATHS, indent=2))

print("COMP_ROOT:", COMP_ROOT)
print("OUT_DIR  :", OUT_DIR)
print("-"*60)

# ----------------------------
# Helpers
# ----------------------------
IMG_EXTS = {".png",".jpg",".jpeg",".bmp",".tif",".tiff",".webp"}

def iter_images(folder: Path, recursive=False):
    if not folder.exists():
        return []
    if recursive:
        files = [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]
    else:
        files = [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS]
    return sorted(files)

def parse_case_id(p: Path):
    # prefer leading digits
    m = re.match(r"^(\d+)", p.stem)
    if m:
        return int(m.group(1))
    # fallback: first digits anywhere
    m2 = re.search(r"(\d+)", p.stem)
    if m2:
        return int(m2.group(1))
    # last resort: keep stem (rare)
    return p.stem

def fast_image_meta(p: Path):
    try:
        with Image.open(p) as im:
            w, h = im.size
            mode = im.mode
        return int(h), int(w), str(mode), None
    except Exception as e:
        return None, None, None, str(e)[:200]

def bbox_from_bool(mask_bool: np.ndarray):
    ys, xs = np.where(mask_bool)
    if xs.size == 0:
        return None
    x1, x2 = int(xs.min()), int(xs.max()) + 1
    y1, y2 = int(ys.min()), int(ys.max()) + 1
    return (x1, y1, x2, y2)

# ----------------------------
# 2) Image profiling
# ----------------------------
img_rows = []

# train: authentic/forged might not exist in some environments; guard gracefully
for p in iter_images(TRAIN_IMG_AUTH, recursive=False):
    img_rows.append({"split":"train","source":"train","label":"authentic","case_id":parse_case_id(p),"img_path":str(p)})

for p in iter_images(TRAIN_IMG_FORG, recursive=False):
    img_rows.append({"split":"train","source":"train","label":"forged","case_id":parse_case_id(p),"img_path":str(p)})

# supplemental: scan non-recursive by default (dataset biasanya flat)
if SUP_IMG_DIR.exists():
    for p in iter_images(SUP_IMG_DIR, recursive=False):
        img_rows.append({"split":"train","source":"supplemental","label":"unknown","case_id":parse_case_id(p),"img_path":str(p)})

# test: bisa flat atau nested => pakai recursive
for p in iter_images(TEST_IMG_DIR, recursive=True):
    img_rows.append({"split":"test","source":"test","label":"unknown","case_id":parse_case_id(p),"img_path":str(p)})

df_img = pd.DataFrame(img_rows)
if len(df_img) == 0:
    raise RuntimeError("No images found. Check COMP_ROOT and folder structure.")

df_img = df_img.drop_duplicates(subset=["split","img_path"]).reset_index(drop=True)
df_img["case_id_str"] = df_img["case_id"].astype(str)

# meta read
metas = []
for p in df_img["img_path"].map(Path):
    h, w, mode, err = fast_image_meta(p)
    metas.append((h, w, mode, err))
df_img[["H","W","mode","img_err"]] = pd.DataFrame(metas, columns=["H","W","mode","img_err"])

# safe derived fields
df_img["aspect"] = np.where(df_img["H"].notna() & (df_img["H"] > 0), df_img["W"] / df_img["H"], np.nan)
df_img["is_gray"] = df_img["mode"].isin(["L","LA","I;16","I"])

# duplicates report (train only)
train_counts = df_img.query("split=='train'").groupby(["case_id_str"]).size()
n_dup_cases = int((train_counts > 1).sum())
if n_dup_cases > 0:
    print("WARNING: duplicate case_id in train split:", n_dup_cases, "cases (handled)")

# Build sizes lookup: (case_id_str, source) -> set((H,W)), plus fallback (case_id_str, 'any')
sizes_lookup = {}
train_hw = df_img.query("split=='train'")[["case_id_str","source","H","W","img_path"]].dropna(subset=["H","W"]).copy()
for r in train_hw.itertuples(index=False):
    key = (r.case_id_str, r.source)
    sizes_lookup.setdefault(key, set()).add((int(r.H), int(r.W)))
    key_any = (r.case_id_str, "any")
    sizes_lookup.setdefault(key_any, set()).add((int(r.H), int(r.W)))

df_img.to_parquet(OUT_DIR / "image_profile.parquet", index=False)

print("IMAGES:")
print(df_img.groupby(["split","source","label"]).size().rename("n").reset_index())
print("Gray ratio (train):", float(df_img.query("split=='train'")["is_gray"].mean()) if len(df_img.query("split=='train'")) else 0.0)
bad_imgs = int(df_img["img_err"].notna().sum())
if bad_imgs:
    print("WARNING: image read errors:", bad_imgs, "(see column img_err)")
print("-"*60)

# Optional: test coverage vs sample_submission
if SAMPLE_SUB.exists():
    try:
        df_ss = pd.read_csv(SAMPLE_SUB)
        if "case_id" in df_ss.columns:
            ss_ids = df_ss["case_id"].astype(str).tolist()
            test_ids = set(df_img.query("split=='test'")["case_id_str"].tolist())
            missing_in_fs = [cid for cid in ss_ids if cid not in test_ids]
            extra_in_fs = [cid for cid in test_ids if cid not in set(ss_ids)]
            if missing_in_fs:
                print("WARNING: test images missing for some sample_submission ids:", len(missing_in_fs))
                print("  Example:", missing_in_fs[:10])
            if extra_in_fs:
                print("NOTE: found extra test images not in sample_submission:", len(extra_in_fs))
    except Exception as e:
        print("WARNING: could not validate sample_submission coverage:", str(e)[:200])

# ----------------------------
# 3) Mask profiling (instance-aware + HWK/KHW safe)
# ----------------------------
mask_files = []
if TRAIN_MASK_DIR.exists():
    mask_files += list(TRAIN_MASK_DIR.glob("*.npy"))
if SUP_MASK_DIR.exists():
    mask_files += list(SUP_MASK_DIR.glob("*.npy"))
mask_files = sorted(mask_files)

mask_rows = []

def iter_instances_from_npy(arr):
    """
    Returns (inst_list, layout_str)
    Supports:
      - (H,W)
      - (K,H,W)
      - (H,W,K)
    """
    if arr.ndim == 2:
        return [arr], "HW"
    if arr.ndim == 3:
        sh = arr.shape
        # heuristic: KHW if first dim small; HWK if last dim small
        if sh[0] <= 32 and sh[1] > 32 and sh[2] > 32:
            return [arr[i] for i in range(sh[0])], "KHW"
        if sh[2] <= 32 and sh[0] > 32 and sh[1] > 32:
            return [arr[:, :, i] for i in range(sh[2])], "HWK"
        # fallback: assume KHW
        return [arr[i] for i in range(sh[0])], "KHW?"
    return None, f"NDIM_{arr.ndim}"

for mp in mask_files:
    case_id = parse_case_id(mp)
    case_id_str = str(case_id)
    src = "train" if mp.parent.name == "train_masks" else "supplemental"

    try:
        arr = np.load(mp, mmap_mode="r")
    except Exception as e:
        mask_rows.append({
            "case_id": case_id, "case_id_str": case_id_str, "source": src,
            "mask_path": str(mp), "raw_ndim": None, "raw_shape": None, "dtype": None, "layout": None,
            "inst_id": -1, "K": None, "H": None, "W": None,
            "area_px": None, "area_frac": None,
            "bbox_x1": None, "bbox_y1": None, "bbox_x2": None, "bbox_y2": None,
            "img_candidates": None, "shape_mismatch": None,
            "mask_err": str(e)[:200]
        })
        continue

    insts, layout = iter_instances_from_npy(arr)
    if insts is None:
        mask_rows.append({
            "case_id": case_id, "case_id_str": case_id_str, "source": src,
            "mask_path": str(mp), "raw_ndim": int(arr.ndim), "raw_shape": str(tuple(arr.shape)), "dtype": str(arr.dtype), "layout": str(layout),
            "inst_id": -1, "K": None, "H": None, "W": None,
            "area_px": None, "area_frac": None,
            "bbox_x1": None, "bbox_y1": None, "bbox_x2": None, "bbox_y2": None,
            "img_candidates": None, "shape_mismatch": None,
            "mask_err": f"Unexpected ndim={arr.ndim}"
        })
        continue

    K = len(insts)
    # determine candidate image sizes for mismatch check
    key = (case_id_str, src)
    cand_sizes = sizes_lookup.get(key, None)
    if cand_sizes is None:
        cand_sizes = sizes_lookup.get((case_id_str, "any"), None)
    img_candidates = int(len(cand_sizes)) if cand_sizes is not None else 0

    for i, m in enumerate(insts):
        try:
            m_arr = np.asarray(m)
            # ensure 2D
            if m_arr.ndim != 2:
                raise ValueError(f"Instance not 2D (ndim={m_arr.ndim})")
            m_bool = (m_arr > 0)
            Hm, Wm = int(m_bool.shape[0]), int(m_bool.shape[1])
            area = int(np.count_nonzero(m_bool))
            bbox = bbox_from_bool(m_bool)
            if bbox is None:
                x1=y1=x2=y2=None
            else:
                x1,y1,x2,y2 = bbox

            # mismatch logic: mismatch=1 if we have candidates AND (Hm,Wm) matches none
            if cand_sizes is None:
                sm = None
            else:
                sm = int((Hm, Wm) not in cand_sizes)

            denom = float(Hm * Wm) if (Hm > 0 and Wm > 0) else None
            area_frac = (area / denom) if denom else None

            mask_rows.append({
                "case_id": case_id, "case_id_str": case_id_str, "source": src,
                "mask_path": str(mp), "raw_ndim": int(arr.ndim), "raw_shape": str(tuple(arr.shape)), "dtype": str(arr.dtype), "layout": str(layout),
                "inst_id": int(i), "K": int(K), "H": Hm, "W": Wm,
                "area_px": area, "area_frac": area_frac,
                "bbox_x1": x1, "bbox_y1": y1, "bbox_x2": x2, "bbox_y2": y2,
                "img_candidates": img_candidates, "shape_mismatch": sm,
                "mask_err": None
            })
        except Exception as e:
            mask_rows.append({
                "case_id": case_id, "case_id_str": case_id_str, "source": src,
                "mask_path": str(mp), "raw_ndim": int(arr.ndim), "raw_shape": str(tuple(arr.shape)), "dtype": str(arr.dtype), "layout": str(layout),
                "inst_id": int(i), "K": int(K), "H": None, "W": None,
                "area_px": None, "area_frac": None,
                "bbox_x1": None, "bbox_y1": None, "bbox_x2": None, "bbox_y2": None,
                "img_candidates": img_candidates, "shape_mismatch": None,
                "mask_err": str(e)[:200]
            })

df_mask_idx = pd.DataFrame(mask_rows)
df_mask_idx.to_parquet(OUT_DIR / "mask_index.parquet", index=False)

valid = df_mask_idx[df_mask_idx["mask_err"].isna()].copy()

if len(valid) == 0:
    # still save empty profile
    agg = pd.DataFrame(columns=[
        "case_id_str","case_id","source",
        "n_instances","sum_area_px","max_inst_area_px","mean_inst_area_px",
        "mean_area_frac","max_area_frac","any_shape_mismatch","any_empty_inst",
        "n_mask_files"
    ])
    agg.to_parquet(OUT_DIR / "mask_profile.parquet", index=False)
else:
    # per case_id/source profiling
    # note: sum_area_px overcounts overlap (profiling only)
    agg = valid.groupby(["case_id_str","case_id","source"]).agg(
        n_instances=("inst_id","count"),
        sum_area_px=("area_px","sum"),
        max_inst_area_px=("area_px","max"),
        mean_inst_area_px=("area_px","mean"),
        mean_area_frac=("area_frac","mean"),
        max_area_frac=("area_frac","max"),
        any_shape_mismatch=("shape_mismatch", lambda x: int(np.nanmax(pd.Series(x).fillna(0))) if len(x) else 0),
        any_empty_inst=("area_px", lambda x: int((pd.Series(x).fillna(0)==0).any()))
    ).reset_index()

    # how many mask files per case/source
    mf = valid.groupby(["case_id_str","case_id","source"])["mask_path"].nunique().rename("n_mask_files").reset_index()
    agg = agg.merge(mf, on=["case_id_str","case_id","source"], how="left")

    agg.to_parquet(OUT_DIR / "mask_profile.parquet", index=False)

print("MASKS:")
print("mask files:", int(len(mask_files)))
print("cases with masks:", int(agg["case_id_str"].nunique()) if len(agg) else 0)
print("instances (valid rows):", int(len(valid)))
print("shape mismatches (cases):", int(agg["any_shape_mismatch"].sum()) if len(agg) else 0)
print("cases with empty instance:", int(agg["any_empty_inst"].sum()) if len(agg) else 0)
bad_masks = int(df_mask_idx["mask_err"].notna().sum())
if bad_masks:
    print("WARNING: mask read/parse errors:", bad_masks, "(see mask_err)")
print("-"*60)

# quantiles snapshot
if len(agg):
    q = agg["max_area_frac"].dropna()
    if len(q):
        qs = q.quantile([0,0.25,0.5,0.75,0.9,0.95,0.99]).to_dict()
        print("max_area_frac quantiles:", {float(k): float(v) for k,v in qs.items()})

# ----------------------------
# 4) Update supplemental labels (for downstream safety)
# ----------------------------
# Default safer behavior: supplemental image without mask => assume authentic
SUPP_NO_MASK_LABEL = "authentic"   # change to "unknown" if you prefer
if SUP_IMG_DIR.exists():
    sup_cases_with_mask = set(agg.loc[agg["source"].eq("supplemental"), "case_id_str"].tolist()) if len(agg) else set()
    m_sup = (df_img["source"].eq("supplemental")) & (df_img["split"].eq("train"))
    if m_sup.any():
        def _lab(cid_str):
            return "forged" if cid_str in sup_cases_with_mask else SUPP_NO_MASK_LABEL
        df_img.loc[m_sup, "label"] = df_img.loc[m_sup, "case_id_str"].map(_lab)
        df_img.to_parquet(OUT_DIR / "image_profile.parquet", index=False)

print("DONE. Saved:")
print(" -", OUT_DIR / "paths.json")
print(" -", OUT_DIR / "image_profile.parquet")
print(" -", OUT_DIR / "mask_index.parquet")
print(" -", OUT_DIR / "mask_profile.parquet")


# Data, Labels, CV & Sanity Guards

In [ ]:
# ============================================================
# STAGE — Data, Labels, CV & Sanity Guards (ONE CELL) — RECOD.ai/LUC
# REVISI FULL (profiling-v2 compatible + safer alignment + safer folds)
#
# Builds:
# - df_train_all: unique case_id (string key), label y, chosen img_path, mask stats
# - df_test: unique case_id, chosen img_path, aligned to sample_submission order
# - folds: StratifiedKFold by y (auto fallback if impossible)
#
# Output:
# - /kaggle/working/recodai_luc_prof/train_manifest.parquet
# - /kaggle/working/recodai_luc_prof/test_manifest.parquet
# - /kaggle/working/recodai_luc_prof/folds.parquet
# - /kaggle/working/recodai_luc_prof/sanity_report.json
# - /kaggle/working/recodai_luc_prof/dup_case_images.csv
# ============================================================

import os, json, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# ----------------------------
# Config
# ----------------------------
N_FOLDS = 5
SEED = 42
STRICT = False  # True => raise on critical issues (recommended after stable)

OUT_DIR = Path("/kaggle/working/recodai_luc_prof")
OUT_DIR.mkdir(parents=True, exist_ok=True)

paths_json = OUT_DIR / "paths.json"
if not paths_json.exists():
    raise FileNotFoundError(f"Missing {paths_json}. Run Dataset Profiling cell first.")

PATHS = json.loads(paths_json.read_text())
COMP_ROOT = Path(PATHS.get("COMP_ROOT", "/kaggle/input"))
SAMPLE_SUB = Path(PATHS.get("SAMPLE_SUB", COMP_ROOT / "sample_submission.csv"))

# Inputs from profiling
img_prof_pq  = OUT_DIR / "image_profile.parquet"
mask_idx_pq  = OUT_DIR / "mask_index.parquet"
mask_prof_pq = OUT_DIR / "mask_profile.parquet"
for p in [img_prof_pq, mask_idx_pq, mask_prof_pq]:
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}. Run Dataset Profiling cell first.")

df_img = pd.read_parquet(img_prof_pq)
df_mask_idx = pd.read_parquet(mask_idx_pq)
df_mask_prof = pd.read_parquet(mask_prof_pq)

# sample_submission defines test ids order
if not SAMPLE_SUB.exists():
    raise FileNotFoundError(f"Missing sample_submission.csv at {SAMPLE_SUB}")
df_sub = pd.read_csv(SAMPLE_SUB)
if "case_id" not in df_sub.columns:
    raise ValueError("sample_submission.csv must contain 'case_id' column")
sub_case_id_str = df_sub["case_id"].astype(str).tolist()

# ----------------------------
# Normalize columns (profiling versions compatibility)
# ----------------------------
if "case_id_str" not in df_img.columns:
    df_img["case_id_str"] = df_img["case_id"].astype(str)

# ensure essentials exist
for col in ["split","source","label","img_path","H","W","mode","is_gray","aspect","img_err"]:
    if col not in df_img.columns:
        df_img[col] = np.nan

if df_mask_prof.empty:
    # keep as empty with expected columns
    pass
else:
    if "case_id_str" not in df_mask_prof.columns:
        if "case_id" in df_mask_prof.columns:
            df_mask_prof["case_id_str"] = df_mask_prof["case_id"].astype(str)
        else:
            # try infer from index if any
            df_mask_prof["case_id_str"] = np.nan

    if "source" not in df_mask_prof.columns:
        df_mask_prof["source"] = "train"

# robust numeric get
def _to_int(x, default=0):
    try:
        if pd.isna(x): return default
        return int(x)
    except Exception:
        return default

def _to_float(x, default=np.nan):
    try:
        if pd.isna(x): return default
        return float(x)
    except Exception:
        return default

# ----------------------------
# Helpers: choose best path per case
# ----------------------------
def _is_good_row(r):
    # prefer no img_err and valid H/W
    ok_err = (pd.isna(r.get("img_err", np.nan)) or str(r.get("img_err","")) == "")
    ok_hw = pd.notna(r.get("H", np.nan)) and pd.notna(r.get("W", np.nan))
    return bool(ok_err and ok_hw)

def _prefer_path(rows: pd.DataFrame, want: str):
    """
    want: "forged" | "authentic" | "any"
    Prefer:
      1) desired folder
      2) good rows (no img_err, has H/W)
      3) stable sort by img_path
    """
    rr = rows.copy()
    rr["__good"] = rr.apply(_is_good_row, axis=1).astype(int)

    def pick_from(df):
        if df.empty: return None
        df = df.sort_values(["__good","img_path"], ascending=[False, True])
        return df.iloc[0]

    if want == "forged":
        cand = rr[rr["img_path"].astype(str).str.contains(r"/forged/|\\forged\\", regex=True)]
        p = pick_from(cand)
        if p is not None: return p

    if want == "authentic":
        cand = rr[rr["img_path"].astype(str).str.contains(r"/authentic/|\\authentic\\", regex=True)]
        p = pick_from(cand)
        if p is not None: return p

    return pick_from(rr)

def resolve_train_case(case_id_str: str, group: pd.DataFrame, has_mask_any: bool):
    labels = group["label"].astype(str).tolist()
    any_forged_folder = any(l == "forged" for l in labels)
    any_auth_folder = any(l == "authentic" for l in labels)

    # label resolution:
    # - if any mask exists => forged
    # - else if any forged folder => forged
    # - else => authentic
    if has_mask_any or any_forged_folder:
        y, label = 1, "forged"
        pick = _prefer_path(group, "forged")
        if pick is None:
            pick = _prefer_path(group, "any")
    else:
        y, label = 0, "authentic"
        pick = _prefer_path(group, "authentic") if any_auth_folder else _prefer_path(group, "any")

    out = {
        "uid": str(case_id_str),
        "case_id": str(case_id_str),
        "y": int(y),
        "label": str(label),
        "img_path": str(pick["img_path"]) if pick is not None else None,
        "source": str(pick.get("source", "train")) if pick is not None else None,
        "H": _to_int(pick.get("H", np.nan), default=-1) if pick is not None else -1,
        "W": _to_int(pick.get("W", np.nan), default=-1) if pick is not None else -1,
        "mode": str(pick.get("mode", "")) if pick is not None and pd.notna(pick.get("mode", np.nan)) else None,
        "is_gray": bool(pick.get("is_gray", False)) if pick is not None else False,
        "aspect": _to_float(pick.get("aspect", np.nan)) if pick is not None else np.nan,
        "n_img_paths": int(len(group)),
        "dup_img_paths": int(len(group) > 1),
        "img_err_any": int(group["img_err"].notna().any()),
    }
    # debug strings (saved to dup_case_images.csv; not inside manifest)
    out["_all_img_paths"] = "|".join(sorted(group["img_path"].astype(str).tolist()))
    out["_all_labels"] = "|".join(sorted(set(labels)))
    return out

def resolve_test_case(case_id_str: str, group: pd.DataFrame):
    pick = _prefer_path(group, "any")
    out = {
        "uid": str(case_id_str),
        "case_id": str(case_id_str),
        "img_path": str(pick["img_path"]) if pick is not None else None,
        "source": str(pick.get("source", "test")) if pick is not None else None,
        "H": _to_int(pick.get("H", np.nan), default=-1) if pick is not None else -1,
        "W": _to_int(pick.get("W", np.nan), default=-1) if pick is not None else -1,
        "mode": str(pick.get("mode", "")) if pick is not None and pd.notna(pick.get("mode", np.nan)) else None,
        "is_gray": bool(pick.get("is_gray", False)) if pick is not None else False,
        "aspect": _to_float(pick.get("aspect", np.nan)) if pick is not None else np.nan,
        "n_img_paths": int(len(group)),
        "dup_img_paths": int(len(group) > 1),
        "img_err_any": int(group["img_err"].notna().any()),
        "_all_img_paths": "|".join(sorted(group["img_path"].astype(str).tolist())),
    }
    return out

# ----------------------------
# Mask availability + stats maps (support both old/new profiling schemas)
# ----------------------------
# Determine mask cases from mask_profile first; fallback to mask_index if profile empty
mask_cases_any = set()
if not df_mask_prof.empty and "case_id_str" in df_mask_prof.columns:
    mask_cases_any = set(df_mask_prof["case_id_str"].dropna().astype(str).unique().tolist())
elif not df_mask_idx.empty and "case_id_str" in df_mask_idx.columns:
    v = df_mask_idx[df_mask_idx["mask_err"].isna()].copy()
    if "case_id_str" in v.columns:
        mask_cases_any = set(v["case_id_str"].dropna().astype(str).unique().tolist())

# Build mask_prof map keyed by (case_id_str, source)
mask_prof_map = None
if not df_mask_prof.empty and {"case_id_str","source"}.issubset(df_mask_prof.columns):
    mask_prof_map = df_mask_prof.set_index(["case_id_str","source"]).copy()
else:
    mask_prof_map = pd.DataFrame().set_index(pd.MultiIndex.from_arrays([[],[]], names=["case_id_str","source"]))

def pick_mask_stats(case_id_str: str):
    """
    Prefer (case_id_str, 'train') then ('supplemental') then any source available.
    Returns (row_series_or_None, mask_source_used_or_None)
    """
    for src in ["train", "supplemental"]:
        key = (case_id_str, src)
        if key in mask_prof_map.index:
            return mask_prof_map.loc[key], src
    # fallback: any source
    try:
        sub = mask_prof_map.loc[(case_id_str, slice(None))]
        if isinstance(sub, pd.DataFrame) and len(sub):
            # prefer larger max_area_frac if many
            if "max_area_frac" in sub.columns:
                rr = sub.sort_values("max_area_frac", ascending=False).iloc[0]
            else:
                rr = sub.iloc[0]
            # find its source (index)
            src = rr.name[1] if isinstance(rr.name, tuple) and len(rr.name) >= 2 else None
            return rr, src
    except Exception:
        pass
    return None, None

def extract_mask_stats(row):
    """
    Compatible with:
      - old: n_instances, union_area_px, max_area_frac, mean_area_frac, any_shape_mismatch, any_empty_inst
      - new: n_instances, sum_area_px, max_area_frac, mean_area_frac, any_shape_mismatch, any_empty_inst
    """
    if row is None:
        return {
            "n_instances": 0,
            "union_area_px": 0,
            "max_area_frac": np.nan,
            "mean_area_frac": np.nan,
            "any_shape_mismatch": 0,
            "any_empty_inst": 0,
            "n_mask_files": 0,
        }
    # area key
    area_px = None
    if "union_area_px" in row.index:
        area_px = row.get("union_area_px", 0)
    elif "sum_area_px" in row.index:
        area_px = row.get("sum_area_px", 0)
    else:
        area_px = 0

    return {
        "n_instances": _to_int(row.get("n_instances", 0)),
        "union_area_px": _to_int(area_px),
        "max_area_frac": _to_float(row.get("max_area_frac", np.nan)),
        "mean_area_frac": _to_float(row.get("mean_area_frac", np.nan)),
        "any_shape_mismatch": _to_int(row.get("any_shape_mismatch", 0)),
        "any_empty_inst": _to_int(row.get("any_empty_inst", 0)),
        "n_mask_files": _to_int(row.get("n_mask_files", 0)),
    }

# ----------------------------
# 1) Build df_train_all (unique case_id)
# ----------------------------
df_train_img = df_img[df_img["split"].astype(str).eq("train")].copy()
if df_train_img.empty:
    raise RuntimeError("No train images found in image_profile.parquet")

train_rows = []
dup_debug_rows = []

for cid_str, g in df_train_img.groupby("case_id_str", sort=True):
    cid_str = str(cid_str)
    has_mask = cid_str in mask_cases_any
    rec = resolve_train_case(cid_str, g, has_mask_any=has_mask)
    train_rows.append(rec)
    if rec["dup_img_paths"] == 1:
        dup_debug_rows.append({
            "case_id": cid_str,
            "labels_seen": rec["_all_labels"],
            "paths_seen": rec["_all_img_paths"]
        })

df_train_all = pd.DataFrame(train_rows)

# attach mask stats (and the source used)
ms_list = []
ms_src_list = []
for cid_str in df_train_all["case_id"].astype(str).tolist():
    r, src_used = pick_mask_stats(cid_str)
    st = extract_mask_stats(r if r is not None else None)
    ms_list.append(st)
    ms_src_list.append(src_used)

df_ms = pd.DataFrame(ms_list)
df_train_all = pd.concat([df_train_all.reset_index(drop=True), df_ms.reset_index(drop=True)], axis=1)
df_train_all["mask_source_used"] = ms_src_list
df_train_all["has_mask"] = (df_train_all["n_instances"].astype(int) > 0).astype(int)

# Guards
bad_forg_no_mask = df_train_all.query("y==1 and has_mask==0")
bad_shape = df_train_all.query("any_shape_mismatch==1")

# ----------------------------
# 2) Build df_test aligned to sample_submission
# ----------------------------
df_test_img = df_img[df_img["split"].astype(str).eq("test")].copy()
if df_test_img.empty:
    raise RuntimeError("No test images found in image_profile.parquet")

test_rows = []
for cid_str, g in df_test_img.groupby("case_id_str", sort=True):
    cid_str = str(cid_str)
    test_rows.append(resolve_test_case(cid_str, g))
df_test = pd.DataFrame(test_rows)

# align to sample_submission order (keep all)
df_test = df_test.set_index("case_id")
test_ids = set(df_test.index.astype(str).tolist())

missing_in_test = [cid for cid in sub_case_id_str if str(cid) not in test_ids]
extra_in_test = [cid for cid in test_ids if cid not in set(map(str, sub_case_id_str))]

if len(missing_in_test) > 0:
    msg = f"Missing {len(missing_in_test)} test case_id(s) from filesystem (first 10: {missing_in_test[:10]})"
    if STRICT:
        raise RuntimeError(msg)
    print("WARNING:", msg)

# reindex to sample order; keep missing rows with NaNs
df_test_ordered = df_test.reindex(list(map(str, sub_case_id_str))).reset_index()

# Add flags to prevent downstream crashes
df_test_ordered["test_missing_img"] = df_test_ordered["img_path"].isna().astype(int)

# ----------------------------
# 3) CV folds (StratifiedKFold with safe fallback)
# ----------------------------
y = df_train_all["y"].astype(int).values
n = len(y)

# Determine feasible folds for stratification
pos = int((y == 1).sum())
neg = int((y == 0).sum())
min_class = min(pos, neg)

folds = np.full(n, -1, dtype=np.int32)

def build_folds_stratified(y, n_splits, seed):
    from sklearn.model_selection import StratifiedKFold
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    out = np.full(len(y), -1, dtype=np.int32)
    for f, (_, va_idx) in enumerate(skf.split(np.zeros_like(y), y)):
        out[va_idx] = f
    return out

def build_folds_kfold(n, n_splits, seed):
    from sklearn.model_selection import KFold
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    out = np.full(n, -1, dtype=np.int32)
    for f, (_, va_idx) in enumerate(kf.split(np.arange(n))):
        out[va_idx] = f
    return out

# Choose method
n_splits = int(N_FOLDS)
method = "StratifiedKFold"
if n_splits < 2:
    n_splits = 2

if min_class < 2:
    # cannot stratify at all
    method = "KFold"
    n_splits = min(max(2, n_splits), n)
    folds = build_folds_kfold(n, n_splits, SEED)
else:
    # reduce folds if not enough positives/negatives
    if min_class < n_splits:
        n_splits = min_class
        if n_splits < 2:
            method = "KFold"
            n_splits = min(max(2, N_FOLDS), n)
            folds = build_folds_kfold(n, n_splits, SEED)
        else:
            folds = build_folds_stratified(y, n_splits, SEED)
    else:
        folds = build_folds_stratified(y, n_splits, SEED)

df_train_all["fold"] = folds.astype(int)

# ----------------------------
# 4) Save artifacts + sanity report
# ----------------------------
# dup report
if dup_debug_rows:
    pd.DataFrame(dup_debug_rows).to_csv(OUT_DIR / "dup_case_images.csv", index=False)
else:
    (OUT_DIR / "dup_case_images.csv").write_text("case_id,labels_seen,paths_seen\n")

# drop debug cols from manifest
df_train_save = df_train_all.drop(columns=["_all_img_paths","_all_labels"], errors="ignore").copy()
df_test_save  = df_test_ordered.drop(columns=["_all_img_paths"], errors="ignore").copy()

df_train_save.to_parquet(OUT_DIR / "train_manifest.parquet", index=False)
df_test_save.to_parquet(OUT_DIR / "test_manifest.parquet", index=False)

df_folds = df_train_save[["case_id","fold","y"]].copy()
df_folds.to_parquet(OUT_DIR / "folds.parquet", index=False)

report = {
    "comp_root": str(COMP_ROOT),
    "n_train_cases": int(df_train_save["case_id"].nunique()),
    "n_test_cases_profiled": int(df_test.shape[0]),
    "n_test_cases_sample_submission": int(len(sub_case_id_str)),
    "n_test_missing_vs_sample_submission": int(len(missing_in_test)),
    "n_test_extra_vs_sample_submission": int(len(extra_in_test)),
    "train_y_mean": float(df_train_save["y"].mean()) if len(df_train_save) else 0.0,
    "train_pos": int(pos),
    "train_neg": int(neg),
    "train_dup_case_images": int((df_train_save["dup_img_paths"]==1).sum()) if "dup_img_paths" in df_train_save.columns else 0,
    "train_img_err_any": int((df_train_save["img_err_any"]==1).sum()) if "img_err_any" in df_train_save.columns else 0,
    "forged_no_mask_cases": int(len(bad_forg_no_mask)),
    "shape_mismatch_cases": int(len(bad_shape)),
    "empty_instance_cases": int((df_train_save["any_empty_inst"]==1).sum()) if "any_empty_inst" in df_train_save.columns else 0,
    "cv_method": method,
    "cv_n_splits": int(n_splits),
    "fold_counts": df_train_save["fold"].value_counts().sort_index().to_dict(),
    "fold_pos_counts": df_train_save.groupby("fold")["y"].sum().astype(int).to_dict(),
}
(OUT_DIR / "sanity_report.json").write_text(json.dumps(report, indent=2))

print("SAVED:")
print(" -", OUT_DIR / "train_manifest.parquet")
print(" -", OUT_DIR / "test_manifest.parquet")
print(" -", OUT_DIR / "folds.parquet")
print(" -", OUT_DIR / "sanity_report.json")
print(" -", OUT_DIR / "dup_case_images.csv")
print("-"*60)
print("SANITY (key):")
print(json.dumps({k: report[k] for k in [
    "n_train_cases","train_y_mean","train_dup_case_images",
    "forged_no_mask_cases","shape_mismatch_cases",
    "n_test_missing_vs_sample_submission","cv_method","cv_n_splits"
]}, indent=2))

if len(bad_forg_no_mask) > 0:
    msg = f"Found forged-labeled cases without masks: {len(bad_forg_no_mask)} (see train_manifest.parquet)"
    if STRICT:
        raise RuntimeError(msg)
    print("WARNING:", msg)

if len(bad_shape) > 0:
    msg = f"Found mask/image shape mismatches: {len(bad_shape)} (see train_manifest.parquet)"
    if STRICT:
        raise RuntimeError(msg)
    print("WARNING:", msg)

if int(df_test_save["test_missing_img"].sum()) > 0:
    msg = f"Test missing images for some sample_submission ids: {int(df_test_save['test_missing_img'].sum())}"
    if STRICT:
        raise RuntimeError(msg)
    print("WARNING:", msg)


# DINOv2 Feature Cache 

In [ ]:
# ============================================================
# STAGE — DINOv2 Feature Cache (CPU-Optimized) (ONE CELL) — REVISI FULL (FAIL-SAFE)
# - Fixed model path: /kaggle/input/dinov2/pytorch/base/1
# - Fixed input size: 518x518  -> token grid 37x37 (patch=14)
#
# Requires:
# - /kaggle/working/recodai_luc_prof/paths.json
# - /kaggle/working/recodai_luc_prof/train_manifest.parquet
# - /kaggle/working/recodai_luc_prof/test_manifest.parquet
#
# Output:
# - /kaggle/working/recodai_luc/cache/dinov2_base_518_cfg_<hash>/{train,test}/{uid}.npz
# - /kaggle/working/recodai_luc/cache/dinov2_base_518_cfg_<hash>/tokens_manifest_{train,test}.parquet
# - /kaggle/working/recodai_luc/cache/dinov2_base_518_cfg_<hash>/cfg.json
#
# Key fixes:
# - Works with manifest using case_id as string (and/or uid)
# - No overwrite when duplicates exist (prefer uid; fallback case_id)
# - Handles missing img_path / NaN safely (records err, never crashes)
# - Robust HF output access (last_hidden_state vs tuple)
# - Checks model.patch_size if available; strict token count check
# ============================================================

import os, gc, json, hashlib, time, re
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch

# ----------------------------
# Config
# ----------------------------
DINO_DIR   = Path("/kaggle/input/dinov2/pytorch/base/1")
IMG_SIZE   = 518
PATCH      = 14
HTOK       = IMG_SIZE // PATCH
WTOK       = IMG_SIZE // PATCH

SAVE_DTYPE = "float16"         # "float16" | "float32"
SAVE_COMPRESSED = True         # True -> np.savez_compressed (smaller, slower); False -> np.savez
BATCH     = 8
FORCE_RECOMPUTE = False        # True -> recompute even if file exists

assert IMG_SIZE % PATCH == 0, "IMG_SIZE must be divisible by PATCH"

# ----------------------------
# Load manifests
# ----------------------------
PROF_DIR   = Path("/kaggle/working/recodai_luc_prof")
paths_json = PROF_DIR / "paths.json"
train_pq   = PROF_DIR / "train_manifest.parquet"
test_pq    = PROF_DIR / "test_manifest.parquet"
for p in [paths_json, train_pq, test_pq]:
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}. Run previous stages first.")

PATHS = json.loads(paths_json.read_text())

df_train = pd.read_parquet(train_pq)
df_test  = pd.read_parquet(test_pq)

# Prefer uid to avoid overwriting duplicates; fallback to case_id
def pick_id_col(df):
    for c in ["uid", "case_id"]:
        if c in df.columns:
            return c
    raise ValueError("Manifest must contain 'uid' or 'case_id' column")

ID_COL_TRAIN = pick_id_col(df_train)
ID_COL_TEST  = pick_id_col(df_test)

# Ensure strings
df_train[ID_COL_TRAIN] = df_train[ID_COL_TRAIN].astype(str)
df_test[ID_COL_TEST]   = df_test[ID_COL_TEST].astype(str)

# img_path existence (may be NaN for missing)
if "img_path" not in df_train.columns or "img_path" not in df_test.columns:
    raise ValueError("Manifests must contain 'img_path' column")

# ----------------------------
# Transformers load (local)
# ----------------------------
from transformers import AutoModel
try:
    from transformers import AutoImageProcessor
    processor = AutoImageProcessor.from_pretrained(str(DINO_DIR), local_files_only=True)
except Exception:
    from transformers import AutoFeatureExtractor
    processor = AutoFeatureExtractor.from_pretrained(str(DINO_DIR), local_files_only=True)

model = AutoModel.from_pretrained(str(DINO_DIR), local_files_only=True)
model.eval()
device = torch.device("cpu")
model.to(device)

# Optional: verify patch size if provided by config
try:
    cfg_patch = int(getattr(model.config, "patch_size", PATCH))
    if cfg_patch != PATCH:
        print(f"WARNING: model.config.patch_size={cfg_patch} != PATCH={PATCH}. Using PATCH={PATCH} for grid expectations.")
except Exception:
    pass

# normalization params
mean = np.array(getattr(processor, "image_mean", [0.485, 0.456, 0.406]), dtype=np.float32)
std  = np.array(getattr(processor, "image_std",  [0.229, 0.224, 0.225]), dtype=np.float32)

# ----------------------------
# Cache dirs (cfg-hashed)
# ----------------------------
CFG = {
    "dino_dir": str(DINO_DIR),
    "img_size": IMG_SIZE,
    "patch": PATCH,
    "htok": HTOK,
    "wtok": WTOK,
    "save_dtype": SAVE_DTYPE,
    "save_compressed": bool(SAVE_COMPRESSED),
    "batch": int(BATCH),
    "normalize_mean": mean.tolist(),
    "normalize_std": std.tolist(),
    "id_col_train": ID_COL_TRAIN,
    "id_col_test": ID_COL_TEST,
}
cfg_id = hashlib.sha1(json.dumps(CFG, sort_keys=True).encode()).hexdigest()[:12]

CACHE_ROOT = Path("/kaggle/working/recodai_luc/cache") / f"dinov2_base_518_cfg_{cfg_id}"
TRAIN_OUT  = CACHE_ROOT / "train"
TEST_OUT   = CACHE_ROOT / "test"
TRAIN_OUT.mkdir(parents=True, exist_ok=True)
TEST_OUT.mkdir(parents=True, exist_ok=True)
(CACHE_ROOT / "cfg.json").write_text(json.dumps(CFG, indent=2))

print("CACHE_ROOT:", CACHE_ROOT)
print("Train rows:", len(df_train), "| Test rows:", len(df_test))
print("ID_COL_TRAIN:", ID_COL_TRAIN, "| ID_COL_TEST:", ID_COL_TEST)

# ----------------------------
# Helpers
# ----------------------------
_SAFE_RE = re.compile(r"[^A-Za-z0-9_.-]+")

def safe_id(x: str) -> str:
    x = str(x)
    x = _SAFE_RE.sub("_", x).strip("_")
    if not x:
        return "EMPTY_ID"
    return x

def is_missing_path(pth) -> bool:
    if pth is None:
        return True
    if isinstance(pth, float) and np.isnan(pth):
        return True
    s = str(pth).strip()
    return (s == "") or (s.lower() == "nan") or (s.lower() == "none")

def load_and_preprocess(p: str) -> np.ndarray:
    # return CHW float32 normalized
    with Image.open(p) as im:
        im = im.convert("RGB")
        im = im.resize((IMG_SIZE, IMG_SIZE), resample=Image.BILINEAR)
        x = np.asarray(im, dtype=np.float32) / 255.0  # HWC
    x = (x - mean) / std
    x = np.transpose(x, (2, 0, 1))  # CHW
    return x

@torch.inference_mode()
def encode_batch(x_bchw: torch.Tensor) -> torch.Tensor:
    out = model(pixel_values=x_bchw)
    h = getattr(out, "last_hidden_state", None)
    if h is None:
        h = out[0]  # tuple-like fallback
    # (B, 1+N, D)
    patch = h[:, 1:, :]  # remove CLS
    Bn, N, D = patch.shape
    expN = HTOK * WTOK
    if N != expN:
        raise RuntimeError(f"Token count mismatch: got N={N}, expected {expN} (IMG_SIZE={IMG_SIZE}, PATCH={PATCH})")
    patch = patch.reshape(Bn, HTOK, WTOK, D)  # (B,HTOK,WTOK,D)
    return patch

def save_npz(dst: Path, tok_hw_d: np.ndarray):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if SAVE_COMPRESSED:
        np.savez_compressed(dst, tok=tok_hw_d)
    else:
        np.savez(dst, tok=tok_hw_d)

def run_cache(df: pd.DataFrame, out_dir: Path, split_name: str, id_col: str) -> pd.DataFrame:
    rows = []
    t0 = time.time()
    miss = 0
    done = 0
    skip = 0
    fail = 0

    ids = df[id_col].astype(str).tolist()
    img_paths = df["img_path"].tolist()

    buf_ids, buf_paths, buf_x = [], [], []

    def flush():
        nonlocal done, skip, fail
        if not buf_x:
            return

        x = torch.from_numpy(np.stack(buf_x, axis=0)).to(device)  # (B,3,H,W) float32
        tok = encode_batch(x).cpu().numpy()

        if SAVE_DTYPE == "float16":
            tok = tok.astype(np.float16)
        elif SAVE_DTYPE == "float32":
            tok = tok.astype(np.float32)
        else:
            raise ValueError("SAVE_DTYPE must be float16 or float32")

        for i in range(tok.shape[0]):
            uid_raw = buf_ids[i]
            uid = safe_id(uid_raw)
            pth = buf_paths[i]
            dst = out_dir / f"{uid}.npz"

            if dst.exists() and (not FORCE_RECOMPUTE):
                skip += 1
            else:
                try:
                    save_npz(dst, tok[i])
                    done += 1
                except Exception as e:
                    fail += 1
                    rows.append({
                        "uid": uid_raw, "uid_safe": uid, "split": split_name,
                        "img_path": pth, "npz_path": None, "htok": HTOK, "wtok": WTOK,
                        "dtype": SAVE_DTYPE, "err": f"save_fail:{str(e)[:200]}"
                    })
                    continue

            rows.append({
                "uid": uid_raw,
                "uid_safe": uid,
                "split": split_name,
                "img_path": pth,
                "npz_path": str(dst),
                "htok": HTOK,
                "wtok": WTOK,
                "dtype": SAVE_DTYPE,
                "err": None
            })

        buf_ids.clear(); buf_paths.clear(); buf_x.clear()
        gc.collect()

    n_total = len(ids)
    for idx, (uid_raw, pth) in enumerate(zip(ids, img_paths), start=1):
        uid = safe_id(uid_raw)

        if is_missing_path(pth):
            miss += 1
            rows.append({
                "uid": uid_raw, "uid_safe": uid, "split": split_name,
                "img_path": None, "npz_path": None,
                "htok": HTOK, "wtok": WTOK, "dtype": SAVE_DTYPE,
                "err": "missing_img_path"
            })
            continue

        p = Path(str(pth))
        if not p.exists():
            miss += 1
            rows.append({
                "uid": uid_raw, "uid_safe": uid, "split": split_name,
                "img_path": str(pth), "npz_path": None,
                "htok": HTOK, "wtok": WTOK, "dtype": SAVE_DTYPE,
                "err": "missing_image"
            })
            continue

        dst = out_dir / f"{uid}.npz"
        if dst.exists() and (not FORCE_RECOMPUTE):
            skip += 1
            rows.append({
                "uid": uid_raw, "uid_safe": uid, "split": split_name,
                "img_path": str(pth), "npz_path": str(dst),
                "htok": HTOK, "wtok": WTOK, "dtype": SAVE_DTYPE,
                "err": None
            })
        else:
            try:
                x = load_and_preprocess(str(pth))
                buf_ids.append(uid_raw)
                buf_paths.append(str(pth))
                buf_x.append(x)
                if len(buf_x) >= BATCH:
                    flush()
            except Exception as e:
                fail += 1
                rows.append({
                    "uid": uid_raw, "uid_safe": uid, "split": split_name,
                    "img_path": str(pth), "npz_path": None,
                    "htok": HTOK, "wtok": WTOK, "dtype": SAVE_DTYPE,
                    "err": f"load_fail:{str(e)[:200]}"
                })

        if idx % 500 == 0:
            elapsed = time.time() - t0
            print(f"[{split_name}] {idx}/{n_total} | done={done} skip={skip} miss={miss} fail={fail} | {elapsed:.1f}s")

    flush()
    elapsed = time.time() - t0
    print(f"[{split_name}] finished | done={done} skip={skip} miss={miss} fail={fail} | {elapsed:.1f}s")

    return pd.DataFrame(rows)

# ----------------------------
# Run caching
# ----------------------------
tok_train = run_cache(df_train, TRAIN_OUT, "train", ID_COL_TRAIN)
tok_test  = run_cache(df_test,  TEST_OUT,  "test",  ID_COL_TEST)

tok_train_path = CACHE_ROOT / "tokens_manifest_train.parquet"
tok_test_path  = CACHE_ROOT / "tokens_manifest_test.parquet"
tok_train.to_parquet(tok_train_path, index=False)
tok_test.to_parquet(tok_test_path, index=False)

# globals for later stages
TOKEN_CACHE_ROOT     = CACHE_ROOT
TOKEN_MANIFEST_TRAIN = tok_train_path
TOKEN_MANIFEST_TEST  = tok_test_path

print("SAVED:")
print(" -", tok_train_path)
print(" -", tok_test_path)
print("Globals:")
print(" - TOKEN_CACHE_ROOT =", TOKEN_CACHE_ROOT)
print(" - TOKEN_MANIFEST_TRAIN =", TOKEN_MANIFEST_TRAIN)
print(" - TOKEN_MANIFEST_TEST  =", TOKEN_MANIFEST_TEST)

# quick summary
def _summ(df, name):
    n = len(df)
    ok = int(df["err"].isna().sum()) if "err" in df.columns else 0
    bad = n - ok
    print(f"[{name}] rows={n} ok={ok} bad={bad}")

_summ(tok_train, "train")
_summ(tok_test, "test")


In [ ]:
# ============================================================
# STAGE — BIND EXISTING DINOv2 TOKEN CACHE (ONE CELL)
# Tujuan: pakai embedding yang sudah ada di /kaggle/input (read-only)
# Output: set globals
#   - TOKEN_CACHE_ROOT
#   - TOKEN_MANIFEST_TRAIN
#   - TOKEN_MANIFEST_TEST
# ============================================================

import os, json, re
from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path("/kaggle/input")

def find_token_cache_candidates():
    cands = []
    # Cari folder yang mengandung tokens_manifest_train.parquet
    for p in BASE.rglob("tokens_manifest_train.parquet"):
        root = p.parent
        # root biasanya: .../dinov2_base_518_cfg_xxx
        cfg = root / "cfg.json"
        train_dir = root / "train"
        test_dir  = root / "test"
        if cfg.exists() and train_dir.exists():
            cands.append(root)
    # fallback: cari dinov2_base_518_cfg_*
    if not cands:
        for p in BASE.rglob("dinov2_base_518_cfg_*"):
            if p.is_dir() and (p/"cfg.json").exists() and (p/"tokens_manifest_train.parquet").exists():
                cands.append(p)
    # unique
    uniq = []
    seen = set()
    for r in cands:
        s = str(r.resolve())
        if s not in seen:
            uniq.append(r); seen.add(s)
    return uniq

def approx_npz_count(d: Path, limit=20000):
    # hitung cepat jumlah .npz di folder (tanpa recursive)
    try:
        return sum(1 for _ in d.glob("*.npz"))
    except Exception:
        return -1

cands = find_token_cache_candidates()
if not cands:
    raise FileNotFoundError(
        "Tidak menemukan tokens_manifest_train.parquet di /kaggle/input. "
        "Pastikan dataset embedding sudah di-add sebagai input."
    )

scored = []
for r in cands:
    train_dir = r / "train"
    test_dir  = r / "test"
    n_train = approx_npz_count(train_dir) if train_dir.exists() else 0
    n_test  = approx_npz_count(test_dir)  if test_dir.exists()  else 0
    mtime = (r/"cfg.json").stat().st_mtime if (r/"cfg.json").exists() else 0
    scored.append((n_train, n_test, mtime, r))

# pilih: train_npz terbanyak, lalu test_npz terbanyak, lalu cfg.json terbaru
scored = sorted(scored, key=lambda x: (x[0], x[1], x[2]), reverse=True)
best = scored[0][3]

TOKEN_CACHE_ROOT = best
TOKEN_MANIFEST_TRAIN = best / "tokens_manifest_train.parquet"
TOKEN_MANIFEST_TEST  = best / "tokens_manifest_test.parquet"

print("PICKED TOKEN_CACHE_ROOT:", TOKEN_CACHE_ROOT)
print(" - TOKEN_MANIFEST_TRAIN:", TOKEN_MANIFEST_TRAIN, "| exists:", TOKEN_MANIFEST_TRAIN.exists())
print(" - TOKEN_MANIFEST_TEST :", TOKEN_MANIFEST_TEST,  "| exists:", TOKEN_MANIFEST_TEST.exists())

# sanity read
df_tr = pd.read_parquet(TOKEN_MANIFEST_TRAIN)
print("train manifest rows:", len(df_tr), "| cols:", list(df_tr.columns)[:12], "...")
if TOKEN_MANIFEST_TEST.exists():
    df_te = pd.read_parquet(TOKEN_MANIFEST_TEST)
    print("test  manifest rows:", len(df_te), "| cols:", list(df_te.columns)[:12], "...")
else:
    df_te = None
    print("WARNING: tokens_manifest_test.parquet tidak ada (RUN_TEST nanti harus False atau handle).")

# check one sample tok shape
sample_npz = None
if "npz_path" in df_tr.columns and df_tr["npz_path"].notna().any():
    sample_npz = Path(df_tr.loc[df_tr["npz_path"].notna(), "npz_path"].iloc[0])
elif (TOKEN_CACHE_ROOT/"train").exists():
    # fallback: ambil file pertama
    files = sorted((TOKEN_CACHE_ROOT/"train").glob("*.npz"))
    sample_npz = files[0] if files else None

if sample_npz is not None and sample_npz.exists():
    z = np.load(sample_npz)
    if "tok" in z.files:
        tok = z["tok"]
        print("sample tok:", sample_npz.name, "| shape:", tok.shape, "| dtype:", tok.dtype)
    else:
        print("WARNING: sample npz tidak punya key 'tok':", sample_npz)
else:
    print("WARNING: tidak bisa ambil sample .npz untuk cek tok.")

print("DONE: globals set -> TOKEN_CACHE_ROOT / TOKEN_MANIFEST_TRAIN / TOKEN_MANIFEST_TEST")


# Robust Matching (Top-k + MNN + Multi-Peak Translation)

In [ ]:
# ============================================================
# STAGE — Robust Matching (Top-k + MNN + Multi-Peak Translation) (ONE CELL)
# REVISI FULL (token-manifest v2 compatible + uid-safe + faster + more guards)
#
# Input: DINOv2 token-grid cache (.npz with key 'tok' -> (Htok,Wtok,D))
# Output (per uid):
#   /kaggle/working/recodai_luc/cache/match_cfg_<hash>/{train,test}/{uid_safe}.npz
#     - peaks_dxy : (P,2) int16  [dx,dy]  (token space)
#     - peak_score: (P,)  int32  (#inlier pairs)
#     - src_masks : (P,Htok,Wtok) uint8
#     - tgt_masks : (P,Htok,Wtok) uint8
#
# Also saves:
#   match_manifest_train.parquet / match_manifest_test.parquet
#
# Notes:
# - CPU-friendly via SimHash grouping + MNN; avoids O(N^2) full KNN.
# - Train split: can run only y==1 to save time.
# - Fixes common errors:
#   * token manifest may use uid / uid_safe instead of case_id
#   * missing npz_path handled (never crash)
#   * stable random projections (created once, not per image)
#   * NMS/peaks robust when histogram empty
# ============================================================

import os, json, hashlib, time, re, gc
from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------
# Config
# ----------------------------
ONLY_FORGED_TRAIN = True   # fast
RUN_TEST = True

SIMHASH_BITS = 12          # fewer bits => larger groups (more recall, slower)
PROJ_DIM = 64              # projection dim for similarity
SEED = 123

SIM_THR = 0.55             # sim threshold for directed NN
MIN_SHIFT = 2              # ignore tiny displacement in token space
PEAKS_TOP = 5              # number of translation peaks to keep
PEAK_INLIER_R = 1          # inlier radius around peak (Chebyshev)
NMS_R = 2                  # NMS radius between peaks (Chebyshev)

FORCE_RECOMPUTE = False

PROF_DIR = Path("/kaggle/working/recodai_luc_prof")
OUT_BASE = Path("/kaggle/working/recodai_luc/cache")
OUT_BASE.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Load manifests + find TOKEN cache root
# ----------------------------
paths_json = PROF_DIR / "paths.json"
train_mani = PROF_DIR / "train_manifest.parquet"
test_mani  = PROF_DIR / "test_manifest.parquet"
for p in [paths_json, train_mani, test_mani]:
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}. Run previous stages first.")

df_train = pd.read_parquet(train_mani)
df_test  = pd.read_parquet(test_mani)

def pick_token_cache_root():
    # 1) globals
    if "TOKEN_CACHE_ROOT" in globals():
        r = Path(str(globals()["TOKEN_CACHE_ROOT"]))
        if r.exists():
            return r
    # 2) auto-search
    cands = sorted(OUT_BASE.glob("dinov2_base_518_cfg_*"))
    cands = [c for c in cands if (c/"cfg.json").exists() and (c/"tokens_manifest_train.parquet").exists()]
    if not cands:
        raise FileNotFoundError("Cannot find DINOv2 token cache under /kaggle/working/recodai_luc/cache. Run DINOv2 Feature Cache stage first.")
    cands = sorted(cands, key=lambda p: (p/"cfg.json").stat().st_mtime, reverse=True)
    return cands[0]

TOKEN_ROOT = pick_token_cache_root()
tok_train_pq = TOKEN_ROOT / "tokens_manifest_train.parquet"
tok_test_pq  = TOKEN_ROOT / "tokens_manifest_test.parquet"
if not tok_train_pq.exists():
    raise FileNotFoundError(f"Missing {tok_train_pq}. Run DINOv2 Feature Cache stage first.")

df_tok_train = pd.read_parquet(tok_train_pq)
df_tok_test  = pd.read_parquet(tok_test_pq) if (RUN_TEST and tok_test_pq.exists()) else pd.DataFrame()

# ----------------------------
# Normalize token manifest cols
# ----------------------------
def ensure_col(df, name, default=None):
    if name not in df.columns:
        df[name] = default
    return df

# prefer uid_safe for filenames; keep uid for join/debug; fallback case_id
def pick_id_cols(df):
    id_col = None
    for c in ["uid", "case_id"]:
        if c in df.columns:
            id_col = c
            break
    if id_col is None:
        raise ValueError("tokens_manifest must contain 'uid' or 'case_id'")
    safe_col = "uid_safe" if "uid_safe" in df.columns else id_col
    return id_col, safe_col

ID_COL_TR, SAFE_COL_TR = pick_id_cols(df_tok_train)
ID_COL_TE, SAFE_COL_TE = pick_id_cols(df_tok_test) if len(df_tok_test) else (None, None)

df_tok_train[ID_COL_TR] = df_tok_train[ID_COL_TR].astype(str)
df_tok_train[SAFE_COL_TR] = df_tok_train[SAFE_COL_TR].astype(str)

if len(df_tok_test):
    df_tok_test[ID_COL_TE] = df_tok_test[ID_COL_TE].astype(str)
    df_tok_test[SAFE_COL_TE] = df_tok_test[SAFE_COL_TE].astype(str)

# Make sure npz_path exists
ensure_col(df_tok_train, "npz_path", None)
if len(df_tok_test):
    ensure_col(df_tok_test, "npz_path", None)

# Merge y into token train manifest (join by uid if possible, else by case_id)
def pick_join_key(df_main):
    for c in ["uid", "case_id"]:
        if c in df_main.columns:
            return c
    return None

join_key_train = pick_join_key(df_train)
# df_train manifest in our revised stage uses uid+case_id strings
if join_key_train is None:
    df_train["uid"] = df_train["case_id"].astype(str)
    join_key_train = "uid"

# tokens may not have case_id if saved as uid only; handle both
if "y" not in df_tok_train.columns:
    df_tok_train["y"] = np.nan

if "uid" in df_tok_train.columns and "uid" in df_train.columns:
    df_tok_train = df_tok_train.merge(df_train[["uid","y"]], on="uid", how="left", suffixes=("","_tr"))
    if "y_tr" in df_tok_train.columns:
        df_tok_train["y"] = df_tok_train["y"].fillna(df_tok_train["y_tr"])
        df_tok_train.drop(columns=["y_tr"], inplace=True, errors="ignore")
elif "case_id" in df_tok_train.columns and "case_id" in df_train.columns:
    df_tok_train = df_tok_train.merge(df_train[["case_id","y"]], on="case_id", how="left", suffixes=("","_tr"))
    if "y_tr" in df_tok_train.columns:
        df_tok_train["y"] = df_tok_train["y"].fillna(df_tok_train["y_tr"])
        df_tok_train.drop(columns=["y_tr"], inplace=True, errors="ignore")

if ONLY_FORGED_TRAIN:
    df_tok_train = df_tok_train[df_tok_train["y"].fillna(0).astype(int).eq(1)].reset_index(drop=True)

# Basic token grid dims (expect consistent)
# fallbacks if empty
if len(df_tok_train) and df_tok_train["htok"].notna().any():
    htok = int(df_tok_train["htok"].dropna().iloc[0])
    wtok = int(df_tok_train["wtok"].dropna().iloc[0])
elif len(df_tok_test) and df_tok_test["htok"].notna().any():
    htok = int(df_tok_test["htok"].dropna().iloc[0])
    wtok = int(df_tok_test["wtok"].dropna().iloc[0])
else:
    raise RuntimeError("Cannot infer token grid dims (htok/wtok) from tokens_manifest.")

print("TOKEN_ROOT:", TOKEN_ROOT)
print("Token grid:", (htok, wtok))
print("Train tokens:", len(df_tok_train), "| Test tokens:", len(df_tok_test) if RUN_TEST else 0)
print("Token ID cols:", (ID_COL_TR, SAFE_COL_TR), "| Test:", (ID_COL_TE, SAFE_COL_TE))

# ----------------------------
# Matching cache dirs
# ----------------------------
CFG = {
    "token_root": str(TOKEN_ROOT),
    "simhash_bits": int(SIMHASH_BITS),
    "proj_dim": int(PROJ_DIM),
    "seed": int(SEED),
    "sim_thr": float(SIM_THR),
    "min_shift": int(MIN_SHIFT),
    "peaks_top": int(PEAKS_TOP),
    "peak_inlier_r": int(PEAK_INLIER_R),
    "nms_r": int(NMS_R),
    "only_forged_train": bool(ONLY_FORGED_TRAIN),
}
cfg_id = hashlib.sha1(json.dumps(CFG, sort_keys=True).encode()).hexdigest()[:12]
MATCH_ROOT = OUT_BASE / f"match_cfg_{cfg_id}"
(MATCH_ROOT / "cfg.json").write_text(json.dumps(CFG, indent=2))
TRAIN_OUT = MATCH_ROOT / "train"
TEST_OUT  = MATCH_ROOT / "test"
TRAIN_OUT.mkdir(parents=True, exist_ok=True)
TEST_OUT.mkdir(parents=True, exist_ok=True)

print("MATCH_ROOT:", MATCH_ROOT)

# ----------------------------
# Core math (precompute RNG projections ONCE)
# ----------------------------
def l2norm(x, eps=1e-8):
    n = np.sqrt((x*x).sum(axis=1, keepdims=True)) + eps
    return x / n

def bitpack_signhash(bits_bool: np.ndarray) -> np.ndarray:
    B = bits_bool.shape[1]
    sig = np.zeros(bits_bool.shape[0], dtype=np.uint32)
    for b in range(B):
        sig |= (bits_bool[:, b].astype(np.uint32) << np.uint32(b))
    return sig

def nms_peaks(hist: np.ndarray, topk: int, r: int):
    H, W = hist.shape
    h = hist.copy()
    peaks = []
    scores = []
    for _ in range(topk):
        idx = int(np.argmax(h))
        sc = int(h.flat[idx])
        if sc <= 0:
            break
        y, x = divmod(idx, W)
        peaks.append((x, y))
        scores.append(sc)
        y0 = max(0, y - r); y1 = min(H, y + r + 1)
        x0 = max(0, x - r); x1 = min(W, x + r + 1)
        h[y0:y1, x0:x1] = 0
    return peaks, scores

def build_masks_from_pairs(pairs_i, pairs_j, peak_dx, peak_dy, H, W, r_inlier):
    yi, xi = pairs_i // W, pairs_i % W
    yj, xj = pairs_j // W, pairs_j % W
    dx = (xj - xi)
    dy = (yj - yi)
    inl = (np.abs(dx - peak_dx) <= r_inlier) & (np.abs(dy - peak_dy) <= r_inlier)
    if not np.any(inl):
        return None, None, 0
    si = pairs_i[inl]
    sj = pairs_j[inl]
    src = np.zeros((H*W,), dtype=np.uint8)
    tgt = np.zeros((H*W,), dtype=np.uint8)
    src[si] = 1
    tgt[sj] = 1
    return src.reshape(H, W), tgt.reshape(H, W), int(inl.sum())

# Precompute projections (D depends on token dim, so we create when first tok loaded)
PROJ_READY = {"D": None, "R_bits": None, "R_proj": None}
_rng = np.random.RandomState(SEED)

def ensure_projections(D):
    if PROJ_READY["D"] == D and PROJ_READY["R_bits"] is not None:
        return
    PROJ_READY["D"] = int(D)
    PROJ_READY["R_bits"] = _rng.randn(D, SIMHASH_BITS).astype(np.float32)
    PROJ_READY["R_proj"] = _rng.randn(D, PROJ_DIM).astype(np.float32)

def robust_match_one(tok_hw_d: np.ndarray):
    H, W, D = tok_hw_d.shape
    N = H * W
    X = tok_hw_d.reshape(N, D).astype(np.float32)

    # normalize token vectors
    X = l2norm(X)

    ensure_projections(D)
    R_bits = PROJ_READY["R_bits"]
    R_proj = PROJ_READY["R_proj"]

    bits = (X @ R_bits) > 0
    sig = bitpack_signhash(bits)

    Xp = X @ R_proj
    Xp = l2norm(Xp)

    # group by signature
    order = np.argsort(sig)
    sig_s = sig[order]
    changes = np.nonzero(sig_s[1:] != sig_s[:-1])[0] + 1
    bounds = np.concatenate(([0], changes, [N]))

    best_j = np.full(N, -1, dtype=np.int32)
    best_s = np.full(N, -1e9, dtype=np.float32)

    for a, b in zip(bounds[:-1], bounds[1:]):
        idx = order[a:b]
        g = len(idx)
        if g < 2:
            continue
        G = Xp[idx]          # (g,PROJ_DIM)
        S = G @ G.T          # (g,g)
        np.fill_diagonal(S, -1e9)
        j_local = np.argmax(S, axis=1)
        s_local = S[np.arange(g), j_local]
        ii = idx
        jj = idx[j_local]
        upd = s_local > best_s[ii]
        best_s[ii[upd]] = s_local[upd]
        best_j[ii[upd]] = jj[upd]

    ok = (best_s >= SIM_THR) & (best_j >= 0)
    ii = np.where(ok)[0]
    jj = best_j[ii]

    # MNN
    mnn = (best_j[jj] == ii)
    ii = ii[mnn]; jj = jj[mnn]
    if len(ii) == 0:
        return {
            "peaks_dxy": np.zeros((0,2), dtype=np.int16),
            "peak_score": np.zeros((0,), dtype=np.int32),
            "src_masks": np.zeros((0,H,W), dtype=np.uint8),
            "tgt_masks": np.zeros((0,H,W), dtype=np.uint8),
        }

    # filter trivial local shifts
    dx = (jj % W) - (ii % W)
    dy = (jj // W) - (ii // W)
    dx = dx.astype(np.int32); dy = dy.astype(np.int32)
    not_small = (np.abs(dx) >= MIN_SHIFT) | (np.abs(dy) >= MIN_SHIFT)
    ii = ii[not_small]; jj = jj[not_small]
    if len(ii) == 0:
        return {
            "peaks_dxy": np.zeros((0,2), dtype=np.int16),
            "peak_score": np.zeros((0,), dtype=np.int32),
            "src_masks": np.zeros((0,H,W), dtype=np.uint8),
            "tgt_masks": np.zeros((0,H,W), dtype=np.uint8),
        }

    # displacement histogram
    dx = (jj % W) - (ii % W)
    dy = (jj // W) - (ii // W)
    dx = dx.astype(np.int32); dy = dy.astype(np.int32)

    dx_min, dx_max = -(W-1), (W-1)
    dy_min, dy_max = -(H-1), (H-1)

    hist = np.zeros((dy_max - dy_min + 1, dx_max - dx_min + 1), dtype=np.int32)
    hx = dx - dx_min
    hy = dy - dy_min

    valid = (hx >= 0) & (hx < hist.shape[1]) & (hy >= 0) & (hy < hist.shape[0])
    if not np.any(valid):
        return {
            "peaks_dxy": np.zeros((0,2), dtype=np.int16),
            "peak_score": np.zeros((0,), dtype=np.int32),
            "src_masks": np.zeros((0,H,W), dtype=np.uint8),
            "tgt_masks": np.zeros((0,H,W), dtype=np.uint8),
        }

    hx = hx[valid]; hy = hy[valid]
    ii2 = ii[valid]; jj2 = jj[valid]
    # faster bincount accumulation
    flat = hy.astype(np.int64) * hist.shape[1] + hx.astype(np.int64)
    bc = np.bincount(flat, minlength=hist.size)
    hist = bc.reshape(hist.shape).astype(np.int32)

    pxy, pscore = nms_peaks(hist, topk=PEAKS_TOP, r=NMS_R)

    peaks_dxy, peak_score, src_masks, tgt_masks = [], [], [], []
    for (px, py), sc in zip(pxy, pscore):
        peak_dx = int(px + dx_min)
        peak_dy = int(py + dy_min)
        src, tgt, ninl = build_masks_from_pairs(ii2, jj2, peak_dx, peak_dy, H, W, PEAK_INLIER_R)
        if ninl <= 0:
            continue
        peaks_dxy.append([peak_dx, peak_dy])
        peak_score.append(ninl)
        src_masks.append(src)
        tgt_masks.append(tgt)

    if len(peaks_dxy) == 0:
        return {
            "peaks_dxy": np.zeros((0,2), dtype=np.int16),
            "peak_score": np.zeros((0,), dtype=np.int32),
            "src_masks": np.zeros((0,H,W), dtype=np.uint8),
            "tgt_masks": np.zeros((0,H,W), dtype=np.uint8),
        }

    return {
        "peaks_dxy": np.asarray(peaks_dxy, dtype=np.int16),
        "peak_score": np.asarray(peak_score, dtype=np.int32),
        "src_masks": np.asarray(src_masks, dtype=np.uint8),
        "tgt_masks": np.asarray(tgt_masks, dtype=np.uint8),
    }

def load_tok(npz_path: str):
    z = np.load(npz_path)
    if "tok" not in z:
        raise KeyError("npz missing key 'tok'")
    tok = z["tok"]
    if tok.ndim != 3:
        raise ValueError(f"tok must be (H,W,D), got shape={tok.shape}")
    return tok

def save_match_npz(dst: Path, pack: dict):
    dst.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(dst,
        peaks_dxy=pack["peaks_dxy"],
        peak_score=pack["peak_score"],
        src_masks=pack["src_masks"],
        tgt_masks=pack["tgt_masks"],
    )

# safe filename
_SAFE_RE = re.compile(r"[^A-Za-z0-9_.-]+")
def safe_id(x: str) -> str:
    x = str(x)
    x = _SAFE_RE.sub("_", x).strip("_")
    return x if x else "EMPTY_ID"

def run_split(df_tok: pd.DataFrame, out_dir: Path, split: str, id_col: str, safe_col: str):
    rows = []
    t0 = time.time()
    done = skip = fail = 0

    n_total = len(df_tok)
    for k, r in df_tok.iterrows():
        uid = str(r.get(id_col, ""))
        uid_safe = safe_id(str(r.get(safe_col, uid)))
        npz_in = r.get("npz_path", None)

        if not isinstance(npz_in, str) or (not Path(npz_in).exists()):
            rows.append({"uid": uid, "uid_safe": uid_safe, "split": split,
                         "tok_npz": npz_in if isinstance(npz_in, str) else None,
                         "match_npz": None, "n_peaks": 0, "best_score": 0,
                         "err": "missing_tok"})
            fail += 1
            continue

        dst = out_dir / f"{uid_safe}.npz"

        if dst.exists() and (not FORCE_RECOMPUTE):
            try:
                z = np.load(dst)
                n_peaks = int(z["peaks_dxy"].shape[0]) if "peaks_dxy" in z else 0
                best_score = int(z["peak_score"].max()) if ("peak_score" in z and len(z["peak_score"]) > 0) else 0
                rows.append({"uid": uid, "uid_safe": uid_safe, "split": split,
                             "tok_npz": str(npz_in), "match_npz": str(dst),
                             "n_peaks": n_peaks, "best_score": best_score, "err": None})
                skip += 1
            except Exception as e:
                rows.append({"uid": uid, "uid_safe": uid_safe, "split": split,
                             "tok_npz": str(npz_in), "match_npz": None,
                             "n_peaks": 0, "best_score": 0, "err": f"read_cached_fail:{str(e)[:200]}"})
                fail += 1
            continue

        try:
            tok = load_tok(str(npz_in))
            pack = robust_match_one(tok)
            save_match_npz(dst, pack)
            n_peaks = int(pack["peaks_dxy"].shape[0])
            best_score = int(pack["peak_score"].max()) if n_peaks > 0 else 0
            rows.append({"uid": uid, "uid_safe": uid_safe, "split": split,
                         "tok_npz": str(npz_in), "match_npz": str(dst),
                         "n_peaks": n_peaks, "best_score": best_score, "err": None})
            done += 1
        except Exception as e:
            rows.append({"uid": uid, "uid_safe": uid_safe, "split": split,
                         "tok_npz": str(npz_in), "match_npz": None,
                         "n_peaks": 0, "best_score": 0, "err": str(e)[:200]})
            fail += 1

        if (k + 1) % 200 == 0:
            print(f"[{split}] {k+1}/{n_total} | done={done} skip={skip} fail={fail} | {time.time()-t0:.1f}s")

    print(f"[{split}] finished | done={done} skip={skip} fail={fail} | {time.time()-t0:.1f}s")
    return pd.DataFrame(rows)

# ----------------------------
# Run
# ----------------------------
match_train = run_split(df_tok_train.reset_index(drop=True), TRAIN_OUT, "train", ID_COL_TR, SAFE_COL_TR)

match_test = pd.DataFrame()
if RUN_TEST and len(df_tok_test):
    match_test = run_split(df_tok_test.reset_index(drop=True), TEST_OUT, "test", ID_COL_TE, SAFE_COL_TE)

# save manifests
mtrain_pq = MATCH_ROOT / "match_manifest_train.parquet"
match_train.to_parquet(mtrain_pq, index=False)
print("SAVED:", mtrain_pq)

mtest_pq = None
if RUN_TEST and len(match_test):
    mtest_pq = MATCH_ROOT / "match_manifest_test.parquet"
    match_test.to_parquet(mtest_pq, index=False)
    print("SAVED:", mtest_pq)

# globals
MATCH_CACHE_ROOT = MATCH_ROOT
MATCH_MANIFEST_TRAIN = mtrain_pq
MATCH_MANIFEST_TEST = mtest_pq

print("Globals:")
print(" - MATCH_CACHE_ROOT =", MATCH_CACHE_ROOT)
print(" - MATCH_MANIFEST_TRAIN =", MATCH_MANIFEST_TRAIN)
print(" - MATCH_MANIFEST_TEST  =", MATCH_MANIFEST_TEST)

# quick summary
def _summ(df, name):
    n = len(df)
    ok = int(df["err"].isna().sum()) if "err" in df.columns else 0
    bad = n - ok
    best = int(df["best_score"].max()) if (n and "best_score" in df.columns) else 0
    print(f"[{name}] rows={n} ok={ok} bad={bad} best_score_max={best}")

_summ(match_train, "train")
if RUN_TEST and len(match_test):
    _summ(match_test, "test")


# Verification, Mask Reconstruction & Postprocess

In [ ]:
# ============================================================
# STAGE — Verification, Mask Reconstruction & Postprocess (ONE CELL) — REVISI FULL v8.1 (UID-SAFE)
# Fuse:
#   (A) Robust Matching proposals (token-space src/tgt masks)
#   (B) Optional mask-model probability map (any 2D) if available
#
# Produce:
#   - per uid_safe: final FULL-RES union mask (original HxW) + scalars
#   - per split: pred_features_{train,test}.csv  (for Gate later)
#
# Robustness goals:
# - KEEP ALL train/test IDs (even if no match/prob => empty mask saved)
# - Works with pipelines that use uid/uid_safe (string) instead of int case_id
# - Safe if match_manifest has duplicates (pick best_score else newest mtime)
# - Maskprob auto-pick supports files named by uid_safe / uid / numeric case_id
# - Prob alignment robust (37x37, 518x518, any 2D -> resize to token grid)
# - Instance split + filtering in token-space BEFORE upsample
# - Saves tok_union for debug; full-res union saved as key "mask"
# ============================================================

import os, json, time, re, gc, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore", category=FutureWarning)

# ----------------------------
# Config
# ----------------------------
PROF_DIR = Path("/kaggle/working/recodai_luc_prof")
OUT_BASE = Path("/kaggle/working/recodai_luc/cache")
OUT_BASE.mkdir(parents=True, exist_ok=True)

# Fusion thresholds (token-space probability)
T1 = 0.55          # confident prob
T0 = 0.35          # weak prob (only allowed near seeds)
SEED_DILATE_IT = 1 # token-space dilation iterations

# Component filtering (token-space)
MIN_TOK_AREA = 2
MAX_TOK_AREA_FRAC = 0.80
MAX_INST_KEEP = 8

# Decide "empty/authentic" guard (token-space)
MIN_PEAK_SCORE_KEEP = 6          # if matching weak, likely authentic
MIN_AREA_FRAC_KEEP = 0.0005      # tiny masks -> drop (token-space area frac)

# Optional: allow "prob-only" masks when no match (default OFF for safety)
PROB_ONLY_ENABLE = False
PROB_ONLY_MIN_AREA_FRAC = 0.003   # token-space
PROB_ONLY_MIN_MEAN_PROB = 0.60

# Save
PRED_DIR = OUT_BASE / "pred_ens"
PRED_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# SciPy optional (faster morphology / CC)
# ----------------------------
try:
    import scipy.ndimage as ndi
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

# ----------------------------
# Load manifests
# ----------------------------
train_pq = PROF_DIR / "train_manifest.parquet"
test_pq  = PROF_DIR / "test_manifest.parquet"
paths_json = PROF_DIR / "paths.json"
for p in [train_pq, test_pq, paths_json]:
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}. Run previous stages first.")

df_train = pd.read_parquet(train_pq).copy()
df_test  = pd.read_parquet(test_pq).copy()
PATHS = json.loads(paths_json.read_text())

def pick_id_col(df):
    for c in ["uid", "case_id"]:
        if c in df.columns:
            return c
    raise ValueError("Manifest must contain 'uid' or 'case_id'.")

ID_COL_TR = pick_id_col(df_train)
ID_COL_TE = pick_id_col(df_test)

df_train[ID_COL_TR] = df_train[ID_COL_TR].astype(str)
df_test[ID_COL_TE]  = df_test[ID_COL_TE].astype(str)

# Ensure there is a "uid_safe" (stable filename key)
_SAFE_RE = re.compile(r"[^A-Za-z0-9_.-]+")
def safe_id(x: str) -> str:
    x = str(x)
    x = _SAFE_RE.sub("_", x).strip("_")
    return x if x else "EMPTY_ID"

for df_ in [df_train, df_test]:
    if "uid_safe" not in df_.columns:
        df_["uid_safe"] = df_[pick_id_col(df_)].map(safe_id)

# Best-effort size fallback
def get_hw_row(r):
    H = getattr(r, "H", None)
    W = getattr(r, "W", None)
    try:
        H = int(H) if (H is not None and not (isinstance(H,float) and np.isnan(H))) else None
        W = int(W) if (W is not None and not (isinstance(W,float) and np.isnan(W))) else None
    except Exception:
        H, W = None, None
    # treat invalid as None
    if H is not None and H <= 0: H = None
    if W is not None and W <= 0: W = None
    return H, W

def try_read_hw_from_image(p):
    try:
        if p and Path(str(p)).exists():
            with Image.open(str(p)) as im:
                w, h = im.size
            return int(h), int(w)
    except Exception:
        pass
    return None, None

# ----------------------------
# Auto-pick latest MATCH cache root
# ----------------------------
def pick_latest_match_root():
    if "MATCH_CACHE_ROOT" in globals():
        r = Path(str(globals()["MATCH_CACHE_ROOT"]))
        if r.exists() and (r / "cfg.json").exists():
            return r
    cands = sorted(OUT_BASE.glob("match_cfg_*"))
    cands = [c for c in cands if (c/"cfg.json").exists() and (c/"match_manifest_train.parquet").exists()]
    if not cands:
        raise FileNotFoundError("Cannot find match_cfg_* under /kaggle/working/recodai_luc/cache. Run Robust Matching stage first.")
    cands = sorted(cands, key=lambda p: (p/"cfg.json").stat().st_mtime, reverse=True)
    return cands[0]

MATCH_ROOT = pick_latest_match_root()
MATCH_CFG  = json.loads((MATCH_ROOT / "cfg.json").read_text())
mtrain_pq  = MATCH_ROOT / "match_manifest_train.parquet"
mtest_pq   = MATCH_ROOT / "match_manifest_test.parquet"

df_mtrain = pd.read_parquet(mtrain_pq) if mtrain_pq.exists() else pd.DataFrame()
df_mtest  = pd.read_parquet(mtest_pq)  if mtest_pq.exists()  else pd.DataFrame()

# Normalize match manifests (uid/uid_safe + path + score)
def normalize_match_df(dfm: pd.DataFrame):
    if dfm is None or len(dfm) == 0:
        return pd.DataFrame(columns=["uid","uid_safe","match_npz","best_score"])
    dfm = dfm.copy()
    # infer uid columns
    if "uid" not in dfm.columns:
        if "case_id" in dfm.columns:
            dfm["uid"] = dfm["case_id"].astype(str)
        else:
            dfm["uid"] = ""
    dfm["uid"] = dfm["uid"].astype(str)

    if "uid_safe" not in dfm.columns:
        dfm["uid_safe"] = dfm["uid"].map(safe_id)
    dfm["uid_safe"] = dfm["uid_safe"].astype(str)

    if "match_npz" not in dfm.columns:
        dfm["match_npz"] = None

    # best score col
    sc = None
    for c in ["best_score","best_peak_score","peak_score_max","max_peak_score","score_max"]:
        if c in dfm.columns:
            sc = c
            break
    if sc is None:
        dfm["best_score"] = np.nan
    else:
        dfm["best_score"] = pd.to_numeric(dfm[sc], errors="coerce")

    return dfm[["uid","uid_safe","match_npz","best_score"]]

df_mtrain = normalize_match_df(df_mtrain)
df_mtest  = normalize_match_df(df_mtest)

print("MATCH_ROOT:", MATCH_ROOT)
print("mtrain:", len(df_mtrain), "| mtest:", len(df_mtest))

# ----------------------------
# Infer PATCH/HTOK/WTOK from TOKEN cache if possible, else from any match file
# ----------------------------
def pick_token_cache_root():
    if "TOKEN_CACHE_ROOT" in globals():
        r = Path(str(globals()["TOKEN_CACHE_ROOT"]))
        if r.exists():
            return r
    cands = sorted(OUT_BASE.glob("dinov2_base_518_cfg_*"))
    cands = [c for c in cands if (c/"cfg.json").exists() and (c/"tokens_manifest_train.parquet").exists()]
    if not cands:
        return None
    cands = sorted(cands, key=lambda p: (p/"cfg.json").stat().st_mtime, reverse=True)
    return cands[0]

TOKEN_ROOT = pick_token_cache_root()
PATCH = 14
HTOK = 37
WTOK = 37
IMG_SIZE = 518

if TOKEN_ROOT is not None and (TOKEN_ROOT/"cfg.json").exists():
    try:
        tc = json.loads((TOKEN_ROOT/"cfg.json").read_text())
        PATCH = int(tc.get("patch", tc.get("patch_size", PATCH)))
        HTOK  = int(tc.get("htok", tc.get("Ht", HTOK)))
        WTOK  = int(tc.get("wtok", tc.get("Wt", WTOK)))
        IMG_SIZE = int(tc.get("img_size", IMG_SIZE))
    except Exception:
        pass

# safety: compute IMG_SIZE if HTOK*PATCH known
if HTOK is not None and WTOK is not None and PATCH is not None:
    IMG_SIZE = int(HTOK * PATCH)

print("TOKEN_ROOT:", TOKEN_ROOT)
print("TOK_CFG :", {"PATCH": PATCH, "HTOK": HTOK, "WTOK": WTOK, "IMG_SIZE": IMG_SIZE})

# ----------------------------
# Pick best match_npz per uid if duplicates exist
# ----------------------------
def pick_best_match_paths(df_match: pd.DataFrame):
    if df_match is None or len(df_match) == 0:
        return {}
    dfm = df_match.copy()
    dfm = dfm[dfm["match_npz"].notna()].copy()
    if len(dfm) == 0:
        return {}

    # keep only existing paths if possible
    def _exists(p):
        try:
            return isinstance(p, str) and Path(p).exists()
        except Exception:
            return False
    dfm["_exists"] = dfm["match_npz"].map(_exists)
    dfm = dfm[dfm["_exists"]].copy()
    if len(dfm) == 0:
        return {}

    # if best_score exists use it
    if "best_score" in dfm.columns and dfm["best_score"].notna().any():
        dfm["best_score"] = pd.to_numeric(dfm["best_score"], errors="coerce").fillna(-1)
        dfm = dfm.sort_values(["uid","best_score"], ascending=[True, False])
        dfm = dfm.drop_duplicates("uid", keep="first")
        return dfm.set_index("uid")["match_npz"].to_dict()

    # else use newest file
    def _mtime(p):
        try:
            return Path(p).stat().st_mtime
        except Exception:
            return -1
    dfm["_mtime"] = dfm["match_npz"].map(_mtime)
    dfm = dfm.sort_values(["uid","_mtime"], ascending=[True, False])
    dfm = dfm.drop_duplicates("uid", keep="first")
    return dfm.set_index("uid")["match_npz"].to_dict()

match_map_train = pick_best_match_paths(df_mtrain)
match_map_test  = pick_best_match_paths(df_mtest)

# ----------------------------
# Optional: auto-pick mask-prob dir (supports {uid_safe}.npz / {uid}.npz / {case_id}.npz)
# ----------------------------
def try_candidates_for_uid(d: Path, uid: str, uid_safe: str):
    cands = [
        d / f"{uid_safe}.npz",
        d / f"{uid}.npz",
    ]
    # numeric fallback
    try:
        ci = int(uid)
        cands.append(d / f"{ci}.npz")
    except Exception:
        pass
    # also allow split subdirs (train/test) commonly used
    for sub in ["train","test","train_all","test_all"]:
        sd = d / sub
        if sd.exists():
            cands.append(sd / f"{uid_safe}.npz")
            cands.append(sd / f"{uid}.npz")
            try:
                ci = int(uid)
                cands.append(sd / f"{ci}.npz")
            except Exception:
                pass
    for p in cands:
        if p.exists():
            return p
    return None

def auto_pick_maskprob_dir(sample_uids, sample_uid_safes):
    roots = []
    roots.append(OUT_BASE)
    roots.append(OUT_BASE / "dino_v2")
    roots.append(OUT_BASE / "mask_prob")
    roots.append(OUT_BASE / "pred_mask")
    roots.append(OUT_BASE / "pred_tok")
    roots.append(OUT_BASE / "seg_prob")
    roots.append(OUT_BASE / "mask_model")
    roots = [r for r in roots if r.exists() and r.is_dir()]

    cands = []
    pats = ["mask_prob*","pred_mask*","pred_tok*","pred_base*","pred_giant*","seg_prob*","maskprob*","prob*","maskdl*","pred_*","cache*"]
    for rt in roots:
        for pat in pats:
            cands += list(rt.glob(pat))
        for d in rt.glob("*"):
            if d.is_dir():
                for pat in pats:
                    cands += list(d.glob(pat))

    uniq = []
    seen = set()
    for d in cands:
        if d.is_dir():
            s = str(d.resolve())
            if s not in seen:
                uniq.append(d); seen.add(s)

    best = None
    best_hit = 0
    for d in uniq:
        hit = 0
        for uid, us in zip(sample_uids, sample_uid_safes):
            if try_candidates_for_uid(d, uid, us) is not None:
                hit += 1
        if hit > best_hit:
            best_hit = hit
            best = d
    return best, best_hit

sample = df_train.head(60)
sample_uids = sample[ID_COL_TR].astype(str).tolist()
sample_uid_safes = sample["uid_safe"].astype(str).tolist()
MASKPROB_DIR, hit = auto_pick_maskprob_dir(sample_uids[:40], sample_uid_safes[:40])
if MASKPROB_DIR is not None and hit >= 5:
    print("MASKPROB_DIR picked:", MASKPROB_DIR, f"(hits on samples={hit})")
else:
    MASKPROB_DIR = None
    print("MASKPROB_DIR: None (no reliable cache found)")

# ----------------------------
# Utils: morphology / CC in token-space
# ----------------------------
def dilate_tok(x_bool, it=1):
    if it <= 0:
        return x_bool
    x = x_bool.astype(bool)
    if _HAS_SCIPY:
        return ndi.binary_dilation(x, iterations=it)
    # fallback: 3x3 max-pool style
    for _ in range(it):
        xp = np.pad(x, 1, mode="constant", constant_values=False)
        y = np.zeros_like(x, dtype=bool)
        for dy in (-1,0,1):
            for dx in (-1,0,1):
                y |= xp[1+dy:1+dy+x.shape[0], 1+dx:1+dx+x.shape[1]]
        x = y
    return x

def label_cc(x_bool):
    x = x_bool.astype(bool)
    if _HAS_SCIPY:
        lab, n = ndi.label(x, structure=np.ones((3,3), dtype=np.uint8))
        return lab, int(n)
    # fallback BFS
    H, W = x.shape
    lab = np.zeros((H,W), dtype=np.int32)
    cur = 0
    for y in range(H):
        for x0 in range(W):
            if (not x[y,x0]) or lab[y,x0] != 0:
                continue
            cur += 1
            stack = [(y,x0)]
            lab[y,x0] = cur
            while stack:
                yy, xx = stack.pop()
                for dy in (-1,0,1):
                    for dx in (-1,0,1):
                        if dy==0 and dx==0:
                            continue
                        ny, nx = yy+dy, xx+dx
                        if 0 <= ny < H and 0 <= nx < W and x[ny,nx] and lab[ny,nx]==0:
                            lab[ny,nx] = cur
                            stack.append((ny,nx))
    return lab, int(cur)

def upsample_tok_to_img(x_hw, patch=14):
    # nearest by repeat: (Ht,Wt)->(Ht*patch, Wt*patch)
    return np.kron(x_hw.astype(np.uint8), np.ones((patch,patch), dtype=np.uint8))

def resize_mask_nearest(x_uint8_01, H, W):
    im = Image.fromarray((x_uint8_01.astype(np.uint8) * 255))
    im = im.resize((int(W), int(H)), resample=Image.NEAREST)
    return (np.asarray(im) > 127).astype(np.uint8)

# ----------------------------
# Load match npz (robust keys)
# ----------------------------
def load_match_npz(p):
    z = np.load(p)
    peaks  = z["peaks_dxy"]  if "peaks_dxy"  in z.files else np.zeros((0,2), np.int16)
    scores = z["peak_score"] if "peak_score" in z.files else np.zeros((0,), np.int32)
    src    = z["src_masks"]  if "src_masks"  in z.files else np.zeros((0,HTOK,WTOK), np.uint8)
    tgt    = z["tgt_masks"]  if "tgt_masks"  in z.files else np.zeros((0,HTOK,WTOK), np.uint8)
    # force correct shape if weird
    if src.ndim != 3: src = np.zeros((0,HTOK,WTOK), np.uint8)
    if tgt.ndim != 3: tgt = np.zeros((0,HTOK,WTOK), np.uint8)
    return peaks, scores, src, tgt

# ----------------------------
# Load maskprob npz (robust keys)
# ----------------------------
def load_maskprob_any(uid: str, uid_safe: str):
    if MASKPROB_DIR is None:
        return None
    p = try_candidates_for_uid(MASKPROB_DIR, uid, uid_safe)
    if p is None:
        return None
    z = np.load(str(p))
    # try common keys
    for k in ["prob_tok","p_tok","prob","p","mask_prob","pred","logits","probs","mask"]:
        if k in z.files:
            return z[k]
    keys = list(z.files)
    return z[keys[0]] if keys else None

def align_prob_to_tok(prob_any):
    if prob_any is None:
        return None
    a = np.asarray(prob_any)

    # squeeze leading singleton dims
    while a.ndim > 2 and a.shape[0] == 1:
        a = a[0]
    if a.ndim != 2:
        return None

    a = a.astype(np.float32)

    # already token grid
    if a.shape == (HTOK, WTOK):
        return a

    # if image-grid divisible by PATCH and matches HTOK/WTOK via block mean
    if a.shape[0] % PATCH == 0 and a.shape[1] % PATCH == 0:
        h = a.shape[0] // PATCH
        w = a.shape[1] // PATCH
        if (h, w) == (HTOK, WTOK):
            return a.reshape(HTOK, PATCH, WTOK, PATCH).mean(axis=(1,3))

    # fallback: resize to token grid (bilinear)
    im = Image.fromarray(a)
    im = im.resize((WTOK, HTOK), resample=Image.BILINEAR)
    return np.asarray(im).astype(np.float32)

# ----------------------------
# Core: build token instances from (match seeds + optional prob)
# ----------------------------
def build_token_instances(src_masks, tgt_masks, peak_score, prob_any):
    has_match = int(len(peak_score) > 0 and src_masks.shape[0] > 0 and tgt_masks.shape[0] > 0)
    best_score = int(np.max(peak_score)) if has_match else 0

    # seed union
    if has_match:
        seed = (src_masks.astype(bool) | tgt_masks.astype(bool)).any(axis=0)
        seed = seed[:HTOK, :WTOK]
    else:
        seed = np.zeros((HTOK, WTOK), dtype=bool)

    # prob_tok
    prob_tok = align_prob_to_tok(prob_any)
    has_prob = int(prob_tok is not None)
    mean_prob = float(np.mean(prob_tok)) if has_prob else np.nan

    # fusion
    if has_prob:
        hard = (prob_tok >= T1)
        soft = (prob_tok >= T0)
        seed_d = dilate_tok(seed, SEED_DILATE_IT)
        fused = hard | (seed_d & soft)

        # optional prob-only if no match
        if (not has_match) and PROB_ONLY_ENABLE:
            hard_area_frac = float(hard.mean())
            if (hard_area_frac >= PROB_ONLY_MIN_AREA_FRAC) and (mean_prob >= PROB_ONLY_MIN_MEAN_PROB):
                fused = hard.copy()
            else:
                fused = np.zeros((HTOK,WTOK), dtype=bool)
    else:
        fused = seed.copy()

    # connected components -> instances with filters
    lab, ncc = label_cc(fused)
    insts, areas = [], []
    for k in range(1, ncc+1):
        m = (lab == k)
        a = int(m.sum())
        if a < MIN_TOK_AREA:
            continue
        if a / float(HTOK*WTOK) > MAX_TOK_AREA_FRAC:
            continue
        insts.append(m.astype(np.uint8))
        areas.append(a)

    if len(insts) == 0:
        return {
            "mask_tok_inst": np.zeros((0,HTOK,WTOK), dtype=np.uint8),
            "mask_tok_union": np.zeros((HTOK,WTOK), dtype=np.uint8),
            "n_inst": 0,
            "area_frac_tok": 0.0,
            "best_peak_score": best_score,
            "mean_prob_tok": mean_prob,
            "has_match": has_match,
            "has_prob": has_prob,
        }

    # keep top-K by area
    order = np.argsort(np.asarray(areas))[::-1][:MAX_INST_KEEP]
    mask_tok_inst = np.stack([insts[i] for i in order], axis=0).astype(np.uint8)
    union = (mask_tok_inst.any(axis=0)).astype(np.uint8)
    area_frac = float(union.mean())

    # final guard: weak match + tiny area -> drop
    if (best_score < MIN_PEAK_SCORE_KEEP) and (area_frac < MIN_AREA_FRAC_KEEP):
        return {
            "mask_tok_inst": np.zeros((0,HTOK,WTOK), dtype=np.uint8),
            "mask_tok_union": np.zeros((HTOK,WTOK), dtype=np.uint8),
            "n_inst": 0,
            "area_frac_tok": 0.0,
            "best_peak_score": best_score,
            "mean_prob_tok": mean_prob,
            "has_match": has_match,
            "has_prob": has_prob,
        }

    return {
        "mask_tok_inst": mask_tok_inst,
        "mask_tok_union": union,
        "n_inst": int(mask_tok_inst.shape[0]),
        "area_frac_tok": area_frac,
        "best_peak_score": best_score,
        "mean_prob_tok": mean_prob,
        "has_match": has_match,
        "has_prob": has_prob,
    }

# ----------------------------
# Run split (always writes npz for every uid_safe)
# ----------------------------
def run_split(df_cases: pd.DataFrame, match_map: dict, split: str, id_col: str):
    out_split = PRED_DIR / split
    out_split.mkdir(parents=True, exist_ok=True)

    rows_feat = []
    t0 = time.time()
    rebuilt = 0
    skipped = 0

    # columns access helper
    cols = set(df_cases.columns)

    for j, row in enumerate(df_cases.itertuples(index=False), start=1):
        uid = str(getattr(row, id_col))
        uid_safe = str(getattr(row, "uid_safe")) if "uid_safe" in cols else safe_id(uid)

        # pick H,W (prefer manifest, fallback try read image size if needed)
        H, W = get_hw_row(row)
        img_path = getattr(row, "img_path", None) if "img_path" in cols else None
        if (H is None or W is None):
            h2, w2 = try_read_hw_from_image(img_path)
            if h2 is not None and w2 is not None:
                H, W = h2, w2
        if H is None or W is None:
            H, W = IMG_SIZE, IMG_SIZE  # ultimate fallback

        dst = out_split / f"{uid_safe}.npz"

        # if exists and looks valid -> skip but still log features
        if dst.exists():
            try:
                z = np.load(dst)
                if ("mask" in z.files) and ("n_inst" in z.files) and ("best_peak_score" in z.files):
                    rows_feat.append({
                        "uid": uid,
                        "uid_safe": uid_safe,
                        "split": split,
                        "y": int(getattr(row, "y")) if ("y" in cols and split=="train") else None,
                        "n_inst": int(z["n_inst"]) if "n_inst" in z.files else 0,
                        "area_frac": float(z["area_frac"]) if "area_frac" in z.files else 0.0,
                        "area_frac_tok": float(z["area_frac_tok"]) if "area_frac_tok" in z.files else 0.0,
                        "best_peak_score": int(z["best_peak_score"]) if "best_peak_score" in z.files else 0,
                        "has_match": int(z["has_match"]) if "has_match" in z.files else 0,
                        "has_prob": int(z["has_prob"]) if "has_prob" in z.files else 0,
                        "mean_prob_tok": float(z["mean_prob_tok"]) if "mean_prob_tok" in z.files else np.nan,
                        "match_exists": int(z["match_exists"]) if "match_exists" in z.files else (1 if uid in match_map else 0),
                        "prob_exists": int(z["prob_exists"]) if "prob_exists" in z.files else (1 if load_maskprob_any(uid, uid_safe) is not None else 0),
                        "npz_path": str(dst),
                    })
                    skipped += 1
                    continue
            except Exception:
                pass  # rebuild

        mp = match_map.get(uid, None)
        match_exists = int(isinstance(mp, str) and Path(mp).exists())
        prob_any = load_maskprob_any(uid, uid_safe)
        prob_exists = int(prob_any is not None)

        if match_exists:
            _, scores, src, tgt = load_match_npz(mp)
            pack_tok = build_token_instances(src, tgt, scores, prob_any)
        else:
            pack_tok = build_token_instances(
                np.zeros((0,HTOK,WTOK), np.uint8),
                np.zeros((0,HTOK,WTOK), np.uint8),
                np.zeros((0,), np.int32),
                prob_any
            )

        # token union -> IMG_SIZE -> original HxW
        tok_union = pack_tok["mask_tok_union"]
        mask_img = upsample_tok_to_img(tok_union, patch=PATCH)          # (HTOK*PATCH, WTOK*PATCH) ~ (518,518)
        mask_full = resize_mask_nearest(mask_img, H, W)                 # (H,W)
        area_frac_full = float(mask_full.mean())

        np.savez_compressed(
            dst,
            mask=mask_full.astype(np.uint8),        # FULL-RES union (HxW)
            tok_union=tok_union.astype(np.uint8),   # token union (HTOKxWTOK) for debug
            H=int(H), W=int(W),
            n_inst=int(pack_tok["n_inst"]),
            area_frac=float(area_frac_full),
            area_frac_tok=float(pack_tok["area_frac_tok"]),
            best_peak_score=int(pack_tok["best_peak_score"]),
            has_match=int(pack_tok["has_match"]),
            has_prob=int(pack_tok["has_prob"]),
            mean_prob_tok=float(pack_tok["mean_prob_tok"]) if np.isfinite(pack_tok["mean_prob_tok"]) else np.nan,
            match_exists=int(match_exists),
            prob_exists=int(prob_exists),
        )

        rows_feat.append({
            "uid": uid,
            "uid_safe": uid_safe,
            "split": split,
            "y": int(getattr(row, "y")) if ("y" in cols and split=="train") else None,
            "n_inst": int(pack_tok["n_inst"]),
            "area_frac": float(area_frac_full),
            "area_frac_tok": float(pack_tok["area_frac_tok"]),
            "best_peak_score": int(pack_tok["best_peak_score"]),
            "has_match": int(pack_tok["has_match"]),
            "has_prob": int(pack_tok["has_prob"]),
            "mean_prob_tok": float(pack_tok["mean_prob_tok"]) if np.isfinite(pack_tok["mean_prob_tok"]) else np.nan,
            "match_exists": int(match_exists),
            "prob_exists": int(prob_exists),
            "npz_path": str(dst),
        })
        rebuilt += 1

        if j % 300 == 0:
            print(f"[{split}] {j}/{len(df_cases)} | rebuilt={rebuilt} skip={skipped} | {time.time()-t0:.1f}s")

    print(f"[{split}] finished | rebuilt={rebuilt} skip={skipped} | {time.time()-t0:.1f}s")
    return pd.DataFrame(rows_feat)

# ----------------------------
# Run train/test (keep all rows)
# ----------------------------
need_train_cols = [ID_COL_TR, "uid_safe"]
need_test_cols  = [ID_COL_TE, "uid_safe"]
for c in ["H","W","y","img_path"]:
    if c in df_train.columns and c not in need_train_cols:
        need_train_cols.append(c)
for c in ["H","W","img_path"]:
    if c in df_test.columns and c not in need_test_cols:
        need_test_cols.append(c)

df_cases_train = df_train[need_train_cols].copy()
df_cases_test  = df_test[need_test_cols].copy()

feat_train = run_split(df_cases_train, match_map_train, "train", ID_COL_TR)
feat_test  = run_split(df_cases_test,  match_map_test,  "test",  ID_COL_TE)

# Save features
feat_train_path = PRED_DIR / "pred_features_train.csv"
feat_test_path  = PRED_DIR / "pred_features_test.csv"
feat_train.to_csv(feat_train_path, index=False)
feat_test.to_csv(feat_test_path, index=False)

print("SAVED:")
print(" -", feat_train_path)
print(" -", feat_test_path)
print("PRED_DIR:", PRED_DIR)

# Globals
PRED_ENS_DIR = PRED_DIR
PRED_FEATURES_TRAIN = feat_train_path
PRED_FEATURES_TEST  = feat_test_path
MATCH_CACHE_ROOT = MATCH_ROOT
MASKPROB_DIR_USED = MASKPROB_DIR
